<a href="https://www.kaggle.com/code/trilucas/semg-db6-ai-v1?scriptVersionId=363370000" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:

from pathlib import Path

root = Path("/kaggle/input")

mat_files = sorted(root.rglob("*.mat"))
print("Số file MAT:", len(mat_files))

for p in mat_files[:5]:
    print(p, f"{p.stat().st_size / 1e6:.1f} MB")

if len(mat_files) == 100:
    print("PASS: Đã tìm thấy 100 file DB6")
else:
    print("CHECK: Số file khác 100, cần kiểm tra dataset")

if mat_files:
    dataset_root = next(
        p for p in mat_files[0].parents
        if p.parent == Path("/kaggle/input/datasets")
    ) if Path("/kaggle/input/datasets") in mat_files[0].parents else None
    print("Dataset root:", dataset_root)


In [ ]:

import os
import sys
import subprocess
from pathlib import Path

assert len(mat_files) == 100, "Kiểm tra lại dataset trước"

# Tìm thư mục chung chứa các file DB6
data_root = Path(os.path.commonpath([
    str(p.parent) for p in mat_files
]))
print("DB6 root:", data_root)

# Clone phiên bản cố định từ GitHub
repo = Path("/kaggle/working/sEMG_AI")
commit = "22e92cdd6ececb163d271ee817587cf5472a3ebd"

if not (repo / ".git").exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/lucas-traumen/sEMG_AI.git",
        str(repo)
    ], check=True)

subprocess.run([
    "git", "-C", str(repo),
    "checkout", "--detach", commit
], check=True)

# Kết nối dataset read-only tới repo
mount = repo / "data/raw/db6_mount"
mount.parent.mkdir(parents=True, exist_ok=True)

if mount.is_symlink():
    assert mount.resolve() == data_root.resolve()
elif not mount.exists():
    mount.symlink_to(data_root, target_is_directory=True)
else:
    raise RuntimeError(f"Đường dẫn đã tồn tại: {mount}")

print("Commit:", subprocess.check_output([
    "git", "-C", str(repo), "rev-parse", "--short", "HEAD"
]).decode().strip())

print("MAT files from mount:", len(list(mount.rglob("*.mat"))))


In [ ]:

import sys
import subprocess
import importlib
from pathlib import Path

repo = Path("/kaggle/working/sEMG_AI")

# Kiểm tra thư viện có sẵn
packages = {
    "numpy": "numpy",
    "scipy": "scipy",
    "pandas": "pandas",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "yaml": "PyYAML",
    "torch": "torch",
}

missing = []

for module, package in packages.items():
    try:
        m = importlib.import_module(module)
        print(f"[OK] {package}: {getattr(m, '__version__', 'installed')}")
    except (ImportError, OSError) as e:
        missing.append(package)
        print(f"[MISSING] {package}: {e}")

if missing:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", *missing],
        check=True
    )
    print("Đã cài thư viện thiếu. Khởi động lại kernel nếu cần.")

# Kiểm tra GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

import subprocess
import sys

def run_stage(args):
    print("\nRUN:", " ".join(args), flush=True)
    subprocess.run(
        [sys.executable, *args],
        cwd="/kaggle/working/sEMG_AI",
        check=True
    )

# A0: Kiểm tra cấu trúc dataset
run_stage([
    "scripts/run_preflight.py",
    "--stage", "inventory"
])

# A1: Kiểm tra tín hiệu, kênh và tạo window
run_stage([
    "scripts/run_preflight.py",
    "--stage", "qc"
])

print("\nPREFLIGHT COMPLETED")


In [ ]:

import sys
import numpy as np
from pathlib import Path
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    classification_report
)

repo = Path("/kaggle/working/sEMG_AI")
sys.path.insert(0, str(repo / "src"))

from semg import stages, dataset, manifest, train

# 1. Load config + Train/Validation manifest
ctx = stages.make_context(
    "configs", repo, "diagnostic_ch01"
)
cfg = ctx.config

windows = dataset.load_trainval(cfg)
entries = stages.load_entries(ctx)

manifest_sha = manifest.read_manifest_sha(
    ctx.manifest_sha_path
)["manifest_sha256"]

# 2. Nạp dữ liệu, chỉ đọc Train và Validation
source = stages.build_source_for(
    entries, ctx=ctx, stage="diagnostic_ch01"
)

cache = dataset.ensure_emg_cache(
    config=cfg,
    source=source,
    files=sorted(windows["file"].unique()),
    manifest_sha=manifest_sha
)

scaler = dataset.load_scaler(cfg, "CH01")

def load_split(split):
    return dataset.channel_arrays(
        config=cfg,
        cache_dir=cache,
        windows=windows,
        channel_id="CH01",
        scaler=scaler,
        split=split
    )

tr = load_split("train")
va = load_split("validation")

print("Train:", tr.x.shape)
print("Validation:", va.x.shape)
print("GPU:", train.pick_device())

# 3. Train mới một seed, không ghi đè checkpoint cũ
result = train.train_one(
    config=cfg,
    channel="CH01",
    model_name="cnn_base",
    seed=11,
    x_train=tr.x,
    y_train=tr.y,
    x_val=va.x,
    y_val=va.y,
    scaler_sha256=scaler.sha256,
    progress=lambda h: print(
        f"Epoch {int(h['epoch']):02d} | "
        f"Train loss: {h['train_loss']:.4f} | "
        f"Val loss: {h['val_loss']:.4f} | "
        f"Val acc: {h['val_accuracy']:.4f}",
        flush=True
    )
)

# 4. Restore checkpoint tốt nhất của lần train này
model = train.restore_model(cfg, {
    "model": "cnn_base",
    "state_dict": result.state_dict
})

device = train.pick_device()

def diagnose(data):
    logits = train.predict_logits(
        model, data.x, device=device
    )
    pred = logits.argmax(axis=1)

    per_subject = [
        float(np.mean(pred[data.subjects == s] ==
                      data.y[data.subjects == s]))
        for s in np.unique(data.subjects)
    ]

    return {
        "accuracy": float(np.mean(pred == data.y)),
        "subject_mean": float(np.mean(per_subject)),
        "balanced_acc": balanced_accuracy_score(
            data.y, pred
        ),
        "macro_f1": f1_score(
            data.y, pred, average="macro"
        ),
        "pred": pred
    }

train_metrics = diagnose(tr)
val_metrics = diagnose(va)

print("\n========== DIAGNOSTIC ==========")
print("Best epoch:", result.best_epoch)

for name, m in [
    ("TRAIN", train_metrics),
    ("VALIDATION", val_metrics)
]:
    print(f"\n{name}")
    print(f"Accuracy: {m['accuracy']:.4%}")
    print(f"Subject mean: {m['subject_mean']:.4%}")
    print(f"Balanced accuracy: {m['balanced_acc']:.4%}")
    print(f"Macro-F1: {m['macro_f1']:.4f}")

print("\nGeneralization gap (subject mean):",
      f"{train_metrics['subject_mean'] - val_metrics['subject_mean']:.4%}")

print("\nVALIDATION PER-CLASS REPORT:")
print(classification_report(
    va.y,
    val_metrics["pred"],
    labels=list(range(8)),
    digits=3,
    zero_division=0
))


In [ ]:

# ============================================================
# sEMG_AI - Multi-scale 1D-CNN Ablation v2
# CH01 | Train D1_T1 | Validation D1_T2
# Seed 11 | 8 classes | No Test data
# ============================================================

import random
import copy
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Cấu hình
# ------------------------------------------------------------

SEED = 11
EPOCHS = 50
PATIENCE = 8
BATCH_SIZE = 128
LR = 1e-3
WEIGHT_DECAY = 1e-4

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

OUT_DIR = Path(
    "/kaggle/working/sEMG_AI/experiments/ablation_v2"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 2. Dữ liệu từ các cell trước
# ------------------------------------------------------------

assert "tr" in globals(), "Chưa có dữ liệu tr"
assert "va" in globals(), "Chưa có dữ liệu va"

X_train = torch.from_numpy(
    np.ascontiguousarray(tr.x, dtype=np.float32)
).unsqueeze(1)

X_val = torch.from_numpy(
    np.ascontiguousarray(va.x, dtype=np.float32)
).unsqueeze(1)

y_train = torch.from_numpy(
    np.asarray(tr.y, dtype=np.int64)
)

y_val = torch.from_numpy(
    np.asarray(va.y, dtype=np.int64)
)

assert X_train.shape[1:] == (1, 400)
assert X_val.shape[1:] == (1, 400)
assert y_train.min() >= 0 and y_train.max() < 8
assert y_val.min() >= 0 and y_val.max() < 8

print("Train:", tuple(X_train.shape))
print("Validation:", tuple(X_val.shape))

train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=DEVICE.type == "cuda"
)

train_eval_loader = DataLoader(
    train_ds,
    batch_size=512,
    shuffle=False
)

val_loader = DataLoader(
    val_ds,
    batch_size=512,
    shuffle=False
)

# ------------------------------------------------------------
# 3. Multi-scale CNN
# Kernel 7 / 15 / 31
# ------------------------------------------------------------

class MultiScaleCNN(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()

        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(
                    1, 16,
                    kernel_size=k,
                    padding=k // 2,
                    bias=False
                ),
                nn.BatchNorm1d(16),
                nn.ReLU()
            )
            for k in (7, 15, 31)
        ])

        self.features = nn.Sequential(
            nn.Conv1d(
                48, 64, 7,
                padding=3, bias=False
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(
                64, 96, 5,
                padding=2, bias=False
            ),
            nn.BatchNorm1d(96),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(
                96, 128, 3,
                padding=1, bias=False
            ),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.AdaptiveAvgPool1d(1),
            nn.Flatten()
        )

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        branches = [
            branch(x)
            for branch in self.branches
        ]

        x = torch.cat(branches, dim=1)
        x = self.features(x)
        return self.classifier(x)


model = MultiScaleCNN(num_classes=8).to(DEVICE)

n_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Model: MultiScaleCNN")
print("Parameters:", f"{n_params:,}")

with torch.no_grad():
    model.eval()
    test_shape = model(
        torch.zeros(2, 1, 400, device=DEVICE)
    ).shape

assert tuple(test_shape) == (2, 8)
print("Shape check:", tuple(test_shape))

# ------------------------------------------------------------
# 4. Optimizer và Loss
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

# ------------------------------------------------------------
# 5. Hàm đánh giá
# ------------------------------------------------------------

@torch.no_grad()
def evaluate_loss_acc(loader):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * len(yb)
        correct += (
            logits.argmax(dim=1) == yb
        ).sum().item()
        total += len(yb)

    return total_loss / total, correct / total


@torch.no_grad()
def get_predictions(loader):
    model.eval()
    predictions = []

    for xb, _ in loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        predictions.append(
            logits.argmax(dim=1).cpu().numpy()
        )

    return np.concatenate(predictions)


# ------------------------------------------------------------
# 6. Training Loop
# ------------------------------------------------------------

best_acc = -1.0
best_loss = float("inf")
best_epoch = 0
best_state = None
bad_epochs = 0
history = []

for epoch in range(1, EPOCHS + 1):

    model.train()
    running_loss = 0.0
    total = 0

    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(yb)
        total += len(yb)

    train_loss = running_loss / total

    val_loss, val_acc = evaluate_loss_acc(
        val_loader
    )

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train loss {train_loss:.4f} | "
        f"Val loss {val_loss:.4f} | "
        f"Val accuracy {val_acc:.4%}",
        flush=True
    )

    improved = (
        val_acc > best_acc
        or (
            val_acc == best_acc
            and val_loss < best_loss
        )
    )

    if improved:
        best_acc = val_acc
        best_loss = val_loss
        best_epoch = epoch

        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= PATIENCE:
        print("Early stopping at epoch", epoch)
        break

# ------------------------------------------------------------
# 7. Restore checkpoint tốt nhất
# ------------------------------------------------------------

model.load_state_dict(best_state)
model.eval()

train_pred = get_predictions(train_eval_loader)
val_pred = get_predictions(val_loader)


def evaluate_metrics(y_true, y_pred, subjects):
    subject_mean = np.mean([
        accuracy_score(
            y_true[subjects == s],
            y_pred[subjects == s]
        )
        for s in np.unique(subjects)
    ])

    return {
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "subject_mean": float(subject_mean),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            )
        )
    }


train_metrics = evaluate_metrics(
    tr.y, train_pred, tr.subjects
)

val_metrics_ms = evaluate_metrics(
    va.y, val_pred, va.subjects
)

# ------------------------------------------------------------
# 8. Hiển thị kết quả
# ------------------------------------------------------------

print("\n========== MULTISCALE CNN ==========")
print("Best epoch:", best_epoch)
print("Parameters:", f"{n_params:,}")

for name, metrics in [
    ("TRAIN", train_metrics),
    ("VALIDATION", val_metrics_ms)
]:
    print(f"\n{name}")
    print(
        f"Accuracy: "
        f"{metrics['accuracy']:.4%}"
    )
    print(
        f"Subject mean: "
        f"{metrics['subject_mean']:.4%}"
    )
    print(
        f"Balanced accuracy: "
        f"{metrics['balanced_accuracy']:.4%}"
    )
    print(
        f"Macro-F1: "
        f"{metrics['macro_f1']:.4f}"
    )

print("\n========== CNN COMPARISON ==========")
print("CNN Base Validation: 40.6796%")
print(
    "MultiScale Validation:",
    f"{val_metrics_ms['subject_mean']:.4%}"
)

print(
    "Difference:",
    f"{(val_metrics_ms['subject_mean'] - 0.406796)*100:+.2f} percentage points"
)

print("\nVALIDATION CLASSIFICATION REPORT:")
print(
    classification_report(
        va.y,
        val_pred,
        labels=list(range(8)),
        digits=3,
        zero_division=0
    )
)

# ------------------------------------------------------------
# 9. Lưu kết quả trong Kaggle working directory
# ------------------------------------------------------------

checkpoint_path = (
    OUT_DIR / "multiscale_CH01_seed11_best.pt"
)

torch.save({
    "model_name": "multiscale",
    "channel": "CH01",
    "seed": SEED,
    "best_epoch": best_epoch,
    "state_dict": best_state,
    "train_metrics": train_metrics,
    "val_metrics": val_metrics_ms
}, checkpoint_path)

pd.DataFrame(history).to_csv(
    OUT_DIR / "multiscale_CH01_seed11_history.csv",
    index=False
)

cm = confusion_matrix(
    va.y, val_pred, labels=list(range(8))
)

pd.DataFrame(cm).to_csv(
    OUT_DIR / "multiscale_CH01_seed11_confusion.csv",
    index=False
)

print("\nSaved checkpoint:", checkpoint_path)
print("Training completed successfully!")


In [ ]:

# ============================================================
# sEMG_AI - 14 CHANNEL CNN EXPERIMENT
# 10 Subjects | D1_T1 Train | D1_T2 Validation
# Baseline CNN architecture | Seed 11
# ============================================================

import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

repo = Path("/kaggle/working/sEMG_AI")
sys.path.insert(0, str(repo / "src"))

from semg import stages, dataset, manifest
from semg import models as semg_models

SEED = 11
EPOCHS = 50
PATIENCE = 8
BATCH = 64
LR = 1e-3
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)

# ------------------------------------------------------------
# 1. Load manifest và cache đã tạo từ QC
# ------------------------------------------------------------

ctx14 = stages.make_context(
    str(repo / "configs"),
    repo,
    "diagnostic_multi14"
)

cfg14 = ctx14.config

windows14 = dataset.load_trainval(cfg14)
entries14 = stages.load_entries(ctx14)

manifest_sha14 = manifest.read_manifest_sha(
    ctx14.manifest_sha_path
)["manifest_sha256"]

source14 = stages.build_source_for(
    entries14,
    ctx=ctx14,
    stage="diagnostic_multi14"
)

cache14 = dataset.ensure_emg_cache(
    config=cfg14,
    source=source14,
    files=sorted(windows14["file"].unique()),
    manifest_sha=manifest_sha14
)

channels14 = dataset.channel_ids_from_map(
    dataset.read_channel_map(cfg14)
)

assert len(channels14) == 14, channels14

print("Channels:", channels14)

# ------------------------------------------------------------
# 2. Ghép 14 kênh theo cùng một window
# ------------------------------------------------------------

def make_14ch_split(split):
    x_multi = None
    y_ref = None
    s_ref = None

    for i, channel in enumerate(channels14):

        scaler = dataset.load_scaler(
            cfg14, channel
        )

        arr = dataset.channel_arrays(
            config=cfg14,
            cache_dir=cache14,
            windows=windows14,
            channel_id=channel,
            scaler=scaler,
            split=split
        )

        if x_multi is None:
            x_multi = np.empty(
                (len(arr), 14, 400),
                dtype=np.float32
            )
            y_ref = arr.y.copy()
            s_ref = arr.subjects.copy()
        else:
            assert np.array_equal(
                y_ref, arr.y
            ), "Label alignment error"

            assert np.array_equal(
                s_ref, arr.subjects
            ), "Subject alignment error"

        x_multi[:, i, :] = arr.x

        print(
            f"{split}: loaded {channel}",
            flush=True
        )

        del arr

    return x_multi, y_ref, s_ref


Xtr14, ytr14, str14 = make_14ch_split("train")
Xva14, yva14, sva14 = make_14ch_split("validation")

print("Train:", Xtr14.shape)
print("Validation:", Xva14.shape)

assert Xtr14.shape == (68978, 14, 400)
assert Xva14.shape == (68854, 14, 400)
assert np.isfinite(Xtr14).all()
assert np.isfinite(Xva14).all()

# ------------------------------------------------------------
# 3. DataLoaders
# ------------------------------------------------------------

train_ds14 = TensorDataset(
    torch.from_numpy(Xtr14),
    torch.from_numpy(ytr14)
)

val_ds14 = TensorDataset(
    torch.from_numpy(Xva14),
    torch.from_numpy(yva14)
)

train_loader14 = DataLoader(
    train_ds14,
    batch_size=BATCH,
    shuffle=True,
    pin_memory=DEVICE.type == "cuda"
)

train_eval_loader14 = DataLoader(
    train_ds14,
    batch_size=256,
    shuffle=False
)

val_loader14 = DataLoader(
    val_ds14,
    batch_size=256,
    shuffle=False
)

# ------------------------------------------------------------
# 4. CNN Base - chỉ thay input từ 1 sang 14 kênh
# ------------------------------------------------------------

model14 = semg_models.build(
    cfg14, "cnn_base"
)

old_conv = model14.conv[0].conv

model14.conv[0].conv = nn.Conv1d(
    in_channels=14,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
    dilation=old_conv.dilation,
    bias=False
)

model14 = model14.to(DEVICE)

with torch.no_grad():
    model14.eval()
    output = model14(
        torch.zeros(2, 14, 400, device=DEVICE)
    )

assert output.shape == (2, 8)

params14 = sum(
    p.numel() for p in model14.parameters()
)

print("Model: cnn_base_14ch")
print("Parameters:", params14)
print("Shape check:", tuple(output.shape))

# ------------------------------------------------------------
# 5. Training
# ------------------------------------------------------------

criterion14 = nn.CrossEntropyLoss()

optimizer14 = torch.optim.Adam(
    model14.parameters(),
    lr=LR,
    weight_decay=1e-4
)

@torch.no_grad()
def evaluate14(loader):
    model14.eval()

    total_loss = 0.0
    correct = 0
    count = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        logits = model14(xb)
        loss = criterion14(logits, yb)

        total_loss += loss.item() * len(yb)
        correct += (
            logits.argmax(1) == yb
        ).sum().item()
        count += len(yb)

    return total_loss / count, correct / count


best_acc14 = -1.0
best_loss14 = float("inf")
best_state14 = None
best_epoch14 = 0
bad_epochs14 = 0
history14 = []

for epoch in range(1, EPOCHS + 1):

    model14.train()

    total_loss = 0.0
    count = 0

    for xb, yb in train_loader14:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer14.zero_grad(set_to_none=True)

        logits = model14(xb)
        loss = criterion14(logits, yb)

        loss.backward()
        optimizer14.step()

        total_loss += loss.item() * len(yb)
        count += len(yb)

    train_loss = total_loss / count
    val_loss, val_acc = evaluate14(val_loader14)

    history14.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train loss {train_loss:.4f} | "
        f"Val loss {val_loss:.4f} | "
        f"Val accuracy {val_acc:.4%}",
        flush=True
    )

    if (
        val_acc > best_acc14 or
        (
            val_acc == best_acc14 and
            val_loss < best_loss14
        )
    ):
        best_acc14 = val_acc
        best_loss14 = val_loss
        best_epoch14 = epoch

        best_state14 = {
            k: v.detach().cpu().clone()
            for k, v in model14.state_dict().items()
        }

        bad_epochs14 = 0
    else:
        bad_epochs14 += 1

    if bad_epochs14 >= PATIENCE:
        print("Early stopping:", epoch)
        break

# ------------------------------------------------------------
# 6. Đánh giá checkpoint tốt nhất
# ------------------------------------------------------------

model14.load_state_dict(best_state14)
model14.eval()

@torch.no_grad()
def predict14(loader):
    predictions = []

    for xb, _ in loader:
        xb = xb.to(DEVICE)

        predictions.append(
            model14(xb).argmax(1).cpu().numpy()
        )

    return np.concatenate(predictions)


pred_train14 = predict14(train_eval_loader14)
pred_val14 = predict14(val_loader14)


def metrics14(y_true, pred, subjects):
    subject_acc = [
        accuracy_score(
            y_true[subjects == s],
            pred[subjects == s]
        )
        for s in np.unique(subjects)
    ]

    return {
        "accuracy": accuracy_score(y_true, pred),
        "subject_mean": float(np.mean(subject_acc)),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, pred
        ),
        "macro_f1": f1_score(
            y_true, pred,
            average="macro",
            zero_division=0
        )
    }


train_metrics14 = metrics14(
    ytr14, pred_train14, str14
)

val_metrics14 = metrics14(
    yva14, pred_val14, sva14
)

print("\n========== CNN 14 CHANNELS ==========")
print("Best epoch:", best_epoch14)

for name, m in [
    ("TRAIN", train_metrics14),
    ("VALIDATION", val_metrics14)
]:
    print(f"\n{name}")
    print(f"Accuracy: {m['accuracy']:.4%}")
    print(f"Subject mean: {m['subject_mean']:.4%}")
    print(f"Balanced: {m['balanced_accuracy']:.4%}")
    print(f"Macro-F1: {m['macro_f1']:.4f}")

print("\n========== COMPARISON ==========")
print("CNN 1 channel: 40.6796%")
print("Multi-scale 1 channel: 40.2172%")
print(
    "CNN 14 channels:",
    f"{val_metrics14['subject_mean']:.4%}"
)

print("\nPER-CLASS REPORT:")
print(classification_report(
    yva14, pred_val14,
    labels=list(range(8)),
    digits=3,
    zero_division=0
))

# ------------------------------------------------------------
# 7. Lưu checkpoint riêng
# ------------------------------------------------------------

out14 = repo / "experiments/ablation_v2"
out14.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_name": "cnn_base_14ch",
    "channels": channels14,
    "seed": SEED,
    "best_epoch": best_epoch14,
    "state_dict": best_state14,
    "config_sha256": cfg14.sha256,
    "manifest_sha256": manifest_sha14,
    "train_metrics": train_metrics14,
    "val_metrics": val_metrics14
}, out14 / "cnn_base_14ch_seed11.pt")

pd.DataFrame(history14).to_csv(
    out14 / "cnn_base_14ch_history.csv",
    index=False
)

pd.DataFrame(
    confusion_matrix(
        yva14, pred_val14,
        labels=list(range(8))
    )
).to_csv(
    out14 / "cnn_base_14ch_confusion.csv",
    index=False
)

print("\nCNN 14-channel experiment completed!")


In [ ]:

import pandas as pd
from pathlib import Path

repo = Path("/kaggle/working/sEMG_AI")

inventory = pd.read_csv(
    repo / "data/manifest/db6_files.csv"
)

windows = pd.read_parquet(
    repo / "data/manifest/trainval_windows.parquet"
)

print("=== DATASET FILES ===")
print(
    inventory.groupby(
        ["role", "day", "session"]
    ).size().to_string()
)

print("\n=== CURRENT TRAIN / VALIDATION ===")
print(
    windows.groupby(
        ["split", "day", "session"]
    ).agg(
        files=("file", "nunique"),
        windows=("file", "size"),
        subjects=("subject", "nunique")
    ).to_string()
)

print("\n=== REPETITIONS BY CLASS ===")
print(
    windows.groupby(
        ["split", "class_id"]
    )["repetition"].nunique().unstack(0)
)

print("\n=== CACHE FILES ===")
for stamp in (repo / "data/interim/emg").glob("*/cache.json"):
    import json
    info = json.loads(stamp.read_text())
    print(stamp.parent.name, len(info.get("files", {})), "files")


In [ ]:

from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score
)

ROOT = Path("/kaggle/working/sEMG_AI")
sys.path.insert(0, str(ROOT / "src"))

from semg import config as config_mod
from semg import dataset as ds
from semg import manifest as mf
from semg import models as nn_models
from semg import train as engine

cfg = config_mod.load(ROOT / "configs")
win = ds.load_trainval(cfg)

manifest_hash = mf.read_manifest_sha(
    cfg.path("manifest_dir") /
    "manifest_sha256.txt"
)["manifest_sha256"]

cache_dir = (
    ds.interim_dir(cfg) /
    ds.cache_signature(cfg, manifest_hash)
)

cache_stamp = cache_dir / "cache.json"

assert cache_stamp.exists(), (
    "Chưa có cache A1. Cần chạy preflight trước."
)

stamp = json.loads(cache_stamp.read_text())

assert stamp["config_sha256"] == cfg.sha256
assert stamp["manifest_sha256"] == manifest_hash

assert all(
    (
        cache_dir /
        (Path(f).stem + ".npy")
    ).exists()
    for f in win["file"].unique()
)

CHANNELS = ds.channel_ids_from_map(
    ds.read_channel_map(cfg)
)

assert len(CHANNELS) == 14
assert "CH09" not in CHANNELS
assert "CH10" not in CHANNELS
assert cfg.models["input"]["channels"] == 1
assert cfg.num_classes == 8

assert len(win[win.split == "train"]) == 68978
assert len(win[win.split == "validation"]) == 68854

assert set(win["split"].unique()) == {
    "train", "validation"
}

for role, session in [
    ("train", 1),
    ("validation", 2)
]:
    part = win[win.split == role]

    assert set(part.day.unique()) == {1}
    assert set(part.session.unique()) == {session}
    assert set(part.subject.unique()) == set(
        range(1, 11)
    )

OUT = (
    ROOT /
    "experiments" /
    "full14_single_channel"
)

OUT.mkdir(parents=True, exist_ok=True)

RUNS_CSV = OUT / "runs.csv"
SUBJECTS_CSV = OUT / "per_subject.csv"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)
print("Channels:", CHANNELS)
print("Train windows:", 68978)
print("Validation windows:", 68854)
print("Cache:", cache_dir)
print("Output:", OUT)
print("Preflight: PASSED")


In [ ]:

RUN_DATA = (
    pd.read_csv(RUNS_CSV).to_dict("records")
    if RUNS_CSV.exists()
    else []
)

SUB_DATA = (
    pd.read_csv(SUBJECTS_CSV).to_dict("records")
    if SUBJECTS_CSV.exists()
    else []
)

run_rows = {
    (r["channel"], int(r["seed"])): r
    for r in RUN_DATA
}

subject_rows = {
    (
        r["channel"],
        int(r["seed"]),
        int(r["subject"])
    ): r
    for r in SUB_DATA
}


def save_csv_atomic(records, path):
    frame = pd.DataFrame(list(records))

    if not frame.empty:
        sort_cols = [
            c for c in ("channel", "seed", "subject")
            if c in frame.columns
        ]
        frame = frame.sort_values(sort_cols)

    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    tmp.replace(path)


def run_channel(channel, seeds):

    scaler = ds.load_scaler(cfg, channel)

    tr = ds.channel_arrays(
        config=cfg,
        cache_dir=cache_dir,
        windows=win,
        channel_id=channel,
        scaler=scaler,
        split="train"
    )

    va = ds.channel_arrays(
        config=cfg,
        cache_dir=cache_dir,
        windows=win,
        channel_id=channel,
        scaler=scaler,
        split="validation"
    )

    assert tr.x.shape == (68978, 400)
    assert va.x.shape == (68854, 400)

    assert np.isfinite(tr.x).all()
    assert np.isfinite(va.x).all()

    assert set(np.unique(tr.y)) == set(range(8))
    assert set(np.unique(va.y)) == set(range(8))

    for seed in seeds:

        key = (channel, int(seed))

        ckpt = (
            OUT /
            "checkpoints" /
            channel /
            f"seed_{seed}.pt"
        )

        if ckpt.is_file():

            payload = engine.load_checkpoint(ckpt)

            assert payload["channel"] == channel
            assert payload["seed"] == seed
            assert payload["model"] == "cnn_base"

            assert (
                payload["config_sha256"] ==
                cfg.sha256
            )

            assert (
                payload["manifest_sha256"] ==
                manifest_hash
            )

            assert (
                payload["scaler_sha256"] ==
                scaler.sha256
            )

            completed = (
                key in run_rows
                and all(
                    (channel, seed, s) in subject_rows
                    for s in range(1, 11)
                )
            )

            if completed:
                print(
                    "REUSE:",
                    channel,
                    "seed", seed,
                    "accuracy",
                    f"{run_rows[key]['val_subject_mean']:.2%}",
                    flush=True
                )
                continue

            net = engine.restore_model(cfg, payload)

        else:

            print(
                "\nTRAIN",
                channel,
                "| seed", seed,
                flush=True
            )

            def progress(h):
                ep = int(h["epoch"])

                if ep == 1 or ep % 5 == 0:
                    print(
                        f"  Epoch {ep:02d} | "
                        f"Train loss {h['train_loss']:.4f} | "
                        f"Val acc {h['val_accuracy']:.2%}",
                        flush=True
                    )

            result = engine.train_one(
                config=cfg,
                channel=channel,
                model_name="cnn_base",
                seed=seed,
                x_train=tr.x,
                y_train=tr.y,
                x_val=va.x,
                y_val=va.y,
                scaler_sha256=scaler.sha256,
                device=DEVICE,
                progress=progress
            )

            engine.save_checkpoint(
                result,
                path=ckpt,
                config_sha256=cfg.sha256,
                manifest_sha256=manifest_hash
            )

            payload = engine.load_checkpoint(ckpt)
            net = engine.restore_model(cfg, payload)

        # Đánh giá checkpoint tốt nhất
        logits = engine.predict_logits(
            net,
            va.x,
            device=DEVICE,
            batch_size=512
        )

        pred = logits.argmax(axis=1)

        per_subject = {}

        for subject in sorted(np.unique(va.subjects)):

            mask = va.subjects == subject

            accuracy = float(
                np.mean(pred[mask] == va.y[mask])
            )

            per_subject[int(subject)] = accuracy

            subject_rows[
                (channel, seed, int(subject))
            ] = {
                "channel": channel,
                "seed": int(seed),
                "subject": int(subject),
                "val_accuracy": accuracy
            }

        subject_mean = float(
            np.mean(list(per_subject.values()))
        )

        run_rows[key] = {
            "channel": channel,
            "seed": int(seed),
            "best_epoch": int(payload["best_epoch"]),
            "val_subject_mean": subject_mean,
            "val_global_accuracy": float(
                np.mean(pred == va.y)
            ),
            "val_balanced_accuracy": float(
                balanced_accuracy_score(va.y, pred)
            ),
            "val_macro_f1": float(
                f1_score(
                    va.y,
                    pred,
                    average="macro",
                    zero_division=0
                )
            ),
            "best_val_loss": float(
                payload["best_val_loss"]
            ),
            "params": int(payload["param_count"]),
            "n_subjects": len(per_subject),
            "config_sha256": cfg.sha256,
            "manifest_sha256": manifest_hash
        }

        # Lưu sau từng seed
        save_csv_atomic(
            run_rows.values(),
            RUNS_CSV
        )

        save_csv_atomic(
            subject_rows.values(),
            SUBJECTS_CSV
        )

        print(
            f"DONE {channel} seed {seed} | "
            f"epoch={payload['best_epoch']} | "
            f"subject-mean={subject_mean:.2%} | "
            f"balanced="
            f"{run_rows[key]['val_balanced_accuracy']:.2%}",
            flush=True
        )

        del net, logits, pred

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    del tr, va


def make_ranking(
    allowed=None,
    expected_seeds=(11, 22, 33)
):

    table = pd.DataFrame(run_rows.values())

    table = table[
        table["seed"].isin(expected_seeds)
    ]

    if allowed is not None:
        table = table[
            table["channel"].isin(allowed)
        ]

    counts = table.groupby("channel")[
        "seed"
    ].nunique()

    good_channels = counts[
        counts == len(expected_seeds)
    ].index

    table = table[
        table["channel"].isin(good_channels)
    ]

    ranking = table.groupby(
        "channel",
        as_index=False
    ).agg(
        mean_val_accuracy=(
            "val_subject_mean", "mean"
        ),
        sd_across_seeds=(
            "val_subject_mean", "std"
        ),
        mean_balanced_accuracy=(
            "val_balanced_accuracy", "mean"
        ),
        mean_macro_f1=(
            "val_macro_f1", "mean"
        ),
        mean_val_loss=(
            "best_val_loss", "mean"
        ),
        n_seeds=(
            "seed", "nunique"
        )
    ).sort_values(
        ["mean_val_accuracy", "mean_val_loss"],
        ascending=[False, True]
    ).reset_index(drop=True)

    ranking.insert(
        0,
        "rank",
        np.arange(1, len(ranking) + 1)
    )

    return ranking


# ==================================================
# RUN: ALL 14 CHANNELS
# ==================================================

for channel in CHANNELS:
    run_channel(channel, (11, 22, 33))

rank3 = make_ranking()

assert len(rank3) == 14, (
    "Chưa hoàn thành 14 kênh × 3 seeds"
)

rank3.to_csv(
    OUT / "ranking_3seeds.csv",
    index=False
)

print("\n=== RANKING 14 CHANNELS / 3 SEEDS ===")

print(
    rank3.to_string(
        index=False,
        formatters={
            "mean_val_accuracy":
                lambda x: f"{x:.2%}",
            "sd_across_seeds":
                lambda x: f"{x:.2%}",
            "mean_balanced_accuracy":
                lambda x: f"{x:.2%}",
            "mean_macro_f1":
                lambda x: f"{x:.4f}"
        }
    )
)

print("\nStage 1 completed.")


In [ ]:

# ==================================================
# CONFIRM BEST CHANNELS WITH 5 SEEDS
# ==================================================

best3 = float(
    rank3.iloc[0]["mean_val_accuracy"]
)

finalists = rank3.loc[
    (rank3["rank"] <= 3) |
    (
        rank3["mean_val_accuracy"] >=
        best3 - 0.005
    ),
    "channel"
].tolist()

print("Confirmation channels:", finalists)
print("Extra seeds:", [44, 55])

for channel in finalists:
    run_channel(channel, (44, 55))

rank5 = make_ranking(
    allowed=finalists,
    expected_seeds=(11, 22, 33, 44, 55)
)

assert len(rank5) == len(finalists)

rank5.to_csv(
    OUT / "ranking_5seeds_finalists.csv",
    index=False
)

selected_channel = str(
    rank5.iloc[0]["channel"]
)

selection = {
    "selected_channel": selected_channel,
    "model": "cnn_base",
    "selection_metric":
        "mean subject accuracy across five seeds",
    "screening_channels": CHANNELS,
    "screening_seeds": [11, 22, 33],
    "finalists": finalists,
    "confirmation_seeds": [11, 22, 33, 44, 55],
    "selected_validation_mean": float(
        rank5.iloc[0]["mean_val_accuracy"]
    ),
    "selected_validation_sd_seed": float(
        rank5.iloc[0]["sd_across_seeds"]
    ),
    "config_sha256": cfg.sha256,
    "manifest_sha256": manifest_hash,
    "test_data_used": False,
    "reserve_data_used": False
}

(
    OUT / "channel_selection_5seed.json"
).write_text(
    json.dumps(selection, indent=2),
    encoding="utf-8"
)

print("\n=== FINAL 5-SEED RANKING ===")

print(
    rank5.to_string(
        index=False,
        formatters={
            "mean_val_accuracy":
                lambda x: f"{x:.2%}",
            "sd_across_seeds":
                lambda x: f"{x:.2%}"
        }
    )
)

print("\nSELECTED CHANNEL:", selected_channel)

print(
    "Validation accuracy:",
    f"{selection['selected_validation_mean']:.2%}"
)

if len(rank5) >= 2:
    margin = (
        rank5.iloc[0]["mean_val_accuracy"] -
        rank5.iloc[1]["mean_val_accuracy"]
    )

    print(
        "Margin over second place:",
        f"{margin * 100:.3f} percentage points"
    )

    if margin < 0.005:
        print(
            "WARNING: Top channels differ by less "
            "than 0.5 percentage points. "
            "Selection is not yet strongly separated."
        )

print("\nResults saved:", OUT)


In [ ]:
# NinaPro DB6 — CH01, full adaptive training, Kaggle ONE CELL.
# Paste this ENTIRE CELL in Kaggle and press Shift+Enter.
# Required: attach Kaggle Dataset trilucas/semg-db6 (and enable GPU).
# D1-D3 source train, D4 tune, D5 held out until final support/query evaluation.
# All code and settings are in this single cell. No .py upload, no terminal command.

#!/usr/bin/env python3
# coding: utf-8
"""NinaPro DB6 single-channel FULL pipeline: source training -> CCA-like features
-> prototypical auxiliary objective -> Reptile -> 0/few-shot final tests.

One invocation, all five days. Does NOT re-select channels. D5 is untouched for
model/feature/hyperparameter selection; for testing, D5 support labels are used
ONLY for allowed few-shot adaptation. See associated README for methodology.

Requires: numpy, scipy, scikit-learn, pandas, matplotlib, torch, tqdm.
Kaggle: attach trilucas/semg-db6 to notebook; run
    !python /kaggle/working/db6_full_adaptation_kaggle.py --channel CH01
"""
from __future__ import annotations
import argparse
import copy
import csv
import gc
import hashlib
import io
import json
import os
import random
import re
import time
import warnings
import zipfile
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score

LABELS = {0: 0, 1: 1, 3: 2, 4: 3, 6: 4, 9: 5, 10: 6, 11: 7}
CLASS_NAMES = ["REST", "Large Diameter", "Adducted Thumb", "Index Finger Extension",
               "Medium Wrap", "Writing Tripod", "Power Sphere", "Precision Sphere"]
PATTERN = re.compile(r"^(S(\d+)_D([1-5])_T([12]))\.mat$", re.I)
N_CLASSES = 8

@dataclass
class Config:
    input_root: str = "/kaggle/input"
    output: str = "/kaggle/working/db6_adaptation_full"
    channel: str = "CH01"  # selected channel; NO channel screening inside this script
    fs: int = 2000
    window: int = 400
    hop: int = 200
    cepstral_k: int = 64  # RETAIN first K cepstral coefficients; not original CCA average
    batch_size: int = 256
    num_workers: int = 0
    seed: int = 11
    epochs: int = 30
    lr: float = 1e-3
    weight_decay: float = 1e-4
    scheduler_patience: int = 3
    scheduler_factor: float = 0.5
    min_lr: float = 1e-6
    early_stop_patience: int = 12
    min_delta: float = 1e-4
    lambda_proto: float = 0.25
    meta_iterations: int = 200
    meta_inner_steps: int = 5
    meta_inner_lr: float = 5e-4
    meta_outer_lr: float = 0.10
    meta_val_every: int = 25
    meta_early_stop_patience: int = 5  # number of meta validations
    meta_support_shots: int = 3
    eval_shots: tuple[int, ...] = (1, 2, 3)
    eval_adapt_steps: int = 5
    eval_adapt_lr: float = 5e-4
    val_query_cap_per_class: int = 120  # validation only; final test query is UNLIMITED
    final_query_cap_per_class: int = 0  # 0=all independent query windows
    max_train_windows_per_class_session: int = 0  # 0=all source windows
    cache: bool = True
    device: str = "auto"


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf8")
    temp.replace(path)


def collect_files(root: Path):
    """Prefer loose .mat over zip members when both representations exist."""
    locations = {}
    for p in sorted(root.rglob("*.mat")):
        m = PATTERN.match(p.name)
        if m:
            locations[p.name] = {"path": str(p), "member": None,
                                 "size": p.stat().st_size}
    for z in sorted(root.rglob("*.zip")):
        with zipfile.ZipFile(z) as f:
            for member in f.infolist():
                name = Path(member.filename).name
                m = PATTERN.match(name)
                if m and name not in locations:
                    locations[name] = {"path": str(z), "member": member.filename,
                                       "size": member.file_size}
    return locations


def check_manifest(locations, out):
    expected = {f"S{s}_D{d}_T{t}.mat" for s in range(1, 11)
                for d in range(1, 6) for t in (1, 2)}
    missing = sorted(expected - set(locations))
    if missing:
        raise RuntimeError(f"Missing DB6 files ({len(missing)}): {missing}")
    rows = []
    for f in sorted(expected):
        m = PATTERN.match(f)
        subject, day, session = map(int, m.groups()[1:])
        size = int(locations[f]["size"])
        role = "source" if day <= 3 else ("validation" if day == 4 else "test")
        status = "excluded" if f == "S2_D2_T2.mat" else "ok"
        rows.append({"file": f, "subject": subject, "day": day, "session": session,
                     "role": role, "bytes": size, "status": status,
                     "reason": "known_truncated_8_class_coverage" if status != "ok" else ""})
    df = pd.DataFrame(rows)
    df.to_csv(out / "manifest.csv", index=False)
    print("Manifest: total 100, source 60 (59 usable), val 20, test 20; exclude S2_D2_T2.mat")
    return df


def read_one(loc):
    keys = ["emg", "restimulus", "rerepetition"]
    if loc["member"] is None:
        mat = loadmat(loc["path"], variable_names=keys)
    else:
        with zipfile.ZipFile(loc["path"]) as zf, zf.open(loc["member"]) as stream:
            mat = loadmat(io.BytesIO(stream.read()), variable_names=keys)
    emg = np.asarray(mat["emg"])
    lab = np.asarray(mat["restimulus"]).ravel()
    rep = np.asarray(mat["rerepetition"]).ravel()
    if emg.ndim != 2 or emg.shape[1] != 16 or len(lab) != len(emg) or len(rep) != len(lab):
        raise ValueError("Invalid .mat dimensions for DB6")
    return emg, lab, rep


def create_windows(emg_col, lab, rep, window, hop):
    """Group IDs: active trials by (class, rerepetition), REST by contiguous segment.
    Support/Query split uses independent *groups* so overlapping windows cannot leak.
    """
    n = len(lab)
    transitions = np.r_[0, np.flatnonzero((lab[1:] != lab[:-1]) | (rep[1:] != rep[:-1])) + 1, n]
    xs, ys, gs = [], [], []
    for segment_id, (lo, hi) in enumerate(zip(transitions[:-1], transitions[1:])):
        raw_label = int(lab[lo])
        if raw_label not in LABELS:
            continue
        cls = LABELS[raw_label]
        length = int(hi - lo)
        if length < window:
            continue
        starts = np.arange(int(lo), int(hi) - window + 1, hop, dtype=np.int64)
        chunk = np.asarray(emg_col[starts[:, None] + np.arange(window)[None, :]], dtype=np.float32)
        xs.append(chunk)
        ys.append(np.full(len(starts), cls, dtype=np.int8))
        # Group active repetitions independently per class; REST by distinct intervals.
        group = (cls * 100000 + int(rep[lo])) if cls and int(rep[lo]) > 0 else (1000000 + segment_id)
        gs.append(np.full(len(starts), group, dtype=np.int32))
    if not xs:
        return (np.zeros((0, window), dtype=np.float32), np.zeros(0, dtype=np.int8),
                np.zeros(0, dtype=np.int32))
    return np.concatenate(xs), np.concatenate(ys), np.concatenate(gs)


def cache_file_path(out, cfg, fname, source_size):
    # Immutable source size is part of cache key, avoiding stale data after file repair.
    return out / "cache" / f"{Path(fname).stem}_{cfg.channel}_w{cfg.window}_h{cfg.hop}_b{source_size}.npz"


def load_windows_file(fname, loc, cfg, out):
    path = cache_file_path(out, cfg, fname, loc["size"])
    if cfg.cache and path.exists():
        with np.load(path) as z:
            return z["x"], z["y"], z["g"]
    emg, labels, reps = read_one(loc)
    idx = int(cfg.channel[2:]) - 1
    col = np.asarray(emg[:, idx], dtype=np.float32)
    if not np.all(np.isfinite(col)):
        raise ValueError(f"Non-finite {cfg.channel} in {fname}")
    x, y, g = create_windows(col, labels, reps, cfg.window, cfg.hop)
    del emg, labels, reps, col
    if not len(x) or len(np.unique(y)) < N_CLASSES:
        raise ValueError(f"{fname} has zero windows or fewer than 8 classes after windowing")
    if cfg.cache:
        path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(path, x=x, y=y, g=g)
    return x, y, g


@dataclass
class Session:
    subject: int
    day: int
    session: int
    x: np.ndarray
    y: np.ndarray
    g: np.ndarray

    @property
    def key(self):
        return (self.subject, self.day, self.session)


def load_sessions(locations, manifest, cfg, out, allowed_days=None):
    sessions = []
    for i, row in manifest.iterrows():
        if row["status"] != "ok" or (allowed_days is not None and int(row["day"]) not in allowed_days):
            continue
        fname = row["file"]
        x, y, g = load_windows_file(fname, locations[fname], cfg, out)
        sessions.append(Session(int(row["subject"]), int(row["day"]),
                                int(row["session"]), x, y, g))
        if (len(sessions) % 10) == 0:
            print(f"Windows cached in current partition: {len(sessions)} files", flush=True)
    return sessions


def partition_session(session, shots, seed, query_cap=0):
    """A K-shot support set and independent query windows for all eight classes.
    Choose ONE window from each of K distinct groups (repetitions/REST intervals),
    and exclude ALL windows belonging to those groups from Query.
    """
    sidx, qidx = [], []
    for cls in range(N_CLASSES):
        # Separate deterministic RNG per class: 1-shot support is nested in 2/3-shot.
        rng = np.random.default_rng(seed + session.subject * 1009 + session.day * 101 + session.session * 17 + cls * 100003)
        idx = np.flatnonzero(session.y == cls)
        groups = np.unique(session.g[idx])
        if len(groups) < shots + 1:
            raise RuntimeError(f"Session {session.key}, class {cls}: only {len(groups)} independent groups; need >= {shots+1}")
        groups = rng.permutation(groups)
        chosen = groups[:shots]
        for gid in chosen:
            cand = idx[session.g[idx] == gid]
            sidx.append(int(rng.choice(cand)))
        remain = idx[~np.isin(session.g[idx], chosen)]
        if query_cap > 0 and len(remain) > query_cap:
            remain = rng.choice(remain, size=query_cap, replace=False)
        qidx.extend(remain.tolist())
    return np.asarray(sidx, dtype=np.int64), np.asarray(qidx, dtype=np.int64)


def cepstrum_vector(x: np.ndarray, k=64):
    """Single-channel cepstral *vector*, NOT original CCA mean-per-electrode.
    400-sample window -> first k non-DC real cepstral coefficients.
    """
    power = np.abs(np.fft.rfft(x.astype(np.float64, copy=False), axis=-1)) ** 2
    ceps = np.fft.irfft(np.log(np.maximum(power, 1e-12)), n=x.shape[-1], axis=-1)
    if k >= x.shape[-1]:
        raise ValueError("cepstral_k must be < window")
    return np.ascontiguousarray(ceps[..., 1: k+1], dtype=np.float32)


@dataclass
class Normalizer:
    mean: np.ndarray
    std: np.ndarray
    def transform(self, x):
        return np.asarray((x-self.mean)/self.std, dtype=np.float32)


def fit_normalizer(xs):
    """Fit only on D1-D3. Inputs: arrays of shape (N,L). Raw uses ONE global scalar
    to retain absolute differences between rest and active contractions; cepstral
    inputs use one mean/std per coefficient, feature-wise.
    """
    if not xs:
        raise ValueError("No source samples")
    n = sum(len(x) for x in xs)
    summed = np.zeros(xs[0].shape[1], dtype=np.float64)
    squares = np.zeros_like(summed)
    for x in xs:
        summed += x.astype(np.float64).sum(axis=0)
        squares += (x.astype(np.float64)**2).sum(axis=0)
    mu = summed / n
    sd = np.sqrt(np.maximum(squares/n - mu**2, 1e-12))
    return Normalizer(mu.astype(np.float32), np.maximum(sd, 1e-6).astype(np.float32))


def make_feature_sessions(sessions, kind, cfg, out):
    if kind == "raw":
        # A single global scaler fitted ONLY on source signals, preserves amplitude.
        cnt, total, sq = 0, 0., 0.
        for s in sessions:
            if s.day <= 3:
                x = s.x.astype(np.float64)
                cnt += x.size; total += x.sum(); sq += np.square(x).sum()
        mu = total / cnt; sd = np.sqrt(max(sq/cnt-mu*mu, 1e-12))
        scaler = Normalizer(np.array(mu, np.float32), np.array(max(sd,1e-6), np.float32))
        result = [Session(s.subject,s.day,s.session,scaler.transform(s.x),s.y,s.g) for s in sessions]
    elif kind == "ceps":
        transformed = [Session(s.subject,s.day,s.session,cepstrum_vector(s.x, cfg.cepstral_k),s.y,s.g)
                       for s in sessions]
        src = [s.x for s in transformed if s.day <= 3]
        scaler = fit_normalizer(src)
        result = [Session(s.subject,s.day,s.session,scaler.transform(s.x),s.y,s.g) for s in transformed]
        del transformed
    else:
        raise ValueError(kind)
    atomic_json(out / f"normalizer_{kind}.json", {
        "mean": np.asarray(scaler.mean).tolist(), "std": np.asarray(scaler.std).tolist(),
        "fit_days": [1,2,3], "feature_kind": kind,
        "warning": "Cepstral vector is a researched single-channel modification, NOT original CCA"
    })
    return result


def normalize_heldout_sessions(sessions, kind, cfg, out):
    """Apply D1-D3-fitted preprocessing without fitting or updating anything on D5."""
    payload=json.loads((out/f"normalizer_{kind}.json").read_text(encoding="utf8"))
    mean=np.asarray(payload["mean"],dtype=np.float32)
    std=np.asarray(payload["std"],dtype=np.float32)
    scaler=Normalizer(mean,std)
    results=[]
    for s in sessions:
        x=s.x if kind=="raw" else cepstrum_vector(s.x,cfg.cepstral_k)
        results.append(Session(s.subject,s.day,s.session,scaler.transform(x),s.y,s.g))
    return results


def get_source_arrays(sessions, cfg):
    xs, ys = [], []
    rng = np.random.default_rng(cfg.seed)
    for s in sessions:
        if s.day > 3:
            continue
        if cfg.max_train_windows_per_class_session > 0:
            idxs=[]
            for cls in range(N_CLASSES):
                cand=np.flatnonzero(s.y==cls)
                n=min(len(cand), cfg.max_train_windows_per_class_session)
                idxs.append(rng.choice(cand, n, replace=False))
            ix=np.sort(np.concatenate(idxs))
        else:
            ix=np.arange(len(s.x))
        xs.append(s.x[ix]); ys.append(s.y[ix])
    return np.ascontiguousarray(np.concatenate(xs), dtype=np.float32), np.concatenate(ys).astype(np.int64)


def make_validation_arrays(sessions, cfg):
    xs,ys=[],[]
    for s in sessions:
        if s.day!=4:
            continue
        _, qi=partition_session(s, max(cfg.eval_shots), cfg.seed,
                                query_cap=cfg.val_query_cap_per_class)
        xs.append(s.x[qi]);ys.append(s.y[qi])
    return np.ascontiguousarray(np.concatenate(xs), dtype=np.float32), np.concatenate(ys).astype(np.int64)


class Block(nn.Module):
    def __init__(self, a,b,k):
        super().__init__()
        self.net=nn.Sequential(nn.Conv1d(a,b,k,padding=k//2,bias=False),
                               nn.BatchNorm1d(b),nn.ReLU(inplace=True),nn.MaxPool1d(2))
    def forward(self,x):
        return self.net(x)


class GestureCNN(nn.Module):
    def __init__(self, length):
        super().__init__()
        self.features=nn.Sequential(Block(1,32,7),Block(32,64,5),Block(64,128,3),Block(128,128,3))
        self.pool=nn.AdaptiveAvgPool1d(1)
        self.embed=nn.Sequential(nn.Flatten(),nn.Linear(128,128),nn.ReLU(),nn.Dropout(0.3),nn.Linear(128,128))
        self.head=nn.Linear(128,N_CLASSES)
    def forward(self,x,return_embedding=False):
        z=self.embed(self.pool(self.features(x)))
        logits=self.head(z)
        return (logits,z) if return_embedding else logits


def proto_loss(z,y):
    # MSE radius around mean of own class; zero for single sample classes.
    loss=z.new_zeros(())
    total=0
    for cls in torch.unique(y):
        ix=(y==cls)
        if int(ix.sum())<2:
            continue
        points=z[ix]
        center=points.mean(dim=0,keepdim=True)
        loss += ((points-center)**2).mean() * int(ix.sum())
        total += int(ix.sum())
    return loss/max(total,1)


def loss_value(logits,z,y,lambda_proto):
    ce=nn.functional.cross_entropy(logits,y)
    return (1-lambda_proto)*ce+lambda_proto*proto_loss(z,y) if lambda_proto>0 else ce


def torch_loader(x,y,cfg,shuffle):
    ds=TensorDataset(torch.from_numpy(np.ascontiguousarray(x,dtype=np.float32)),
                     torch.from_numpy(np.asarray(y,dtype=np.int64)))
    return DataLoader(ds,batch_size=cfg.batch_size,shuffle=shuffle,num_workers=cfg.num_workers,
                      pin_memory=torch.cuda.is_available())


@torch.no_grad()
def eval_celoss(model, loader, device):
    model.eval(); n=0; total=0.;correct=0
    for x,y in loader:
        x=x.to(device,non_blocking=True).unsqueeze(1)
        y=y.to(device,non_blocking=True)
        pred=model(x)
        total+=float(nn.functional.cross_entropy(pred,y,reduction="sum"))
        correct+=int((pred.argmax(-1)==y).sum());n+=len(y)
    # summed CE above already sums: do not multiply by N again
    return total/max(n,1),correct/max(n,1)


def write_checkpoint(path,model,meta):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(".tmp")
    torch.save({"state_dict":{k:v.detach().cpu() for k,v in model.state_dict().items()},"meta":meta},tmp)
    tmp.replace(path)


def fit_model(name,feature_sessions,cfg,out,device,lambda_proto):
    x,y=get_source_arrays(feature_sessions,cfg)
    vx,vy=make_validation_arrays(feature_sessions,cfg)
    print(f"[{name}] source {x.shape}, D4 query validation {vx.shape}",flush=True)
    train_loader=torch_loader(x,y,cfg,True);val_loader=torch_loader(vx,vy,cfg,False)
    model=GestureCNN(x.shape[1]).to(device)
    optim=torch.optim.Adam(model.parameters(),lr=cfg.lr,weight_decay=cfg.weight_decay)
    sched=torch.optim.lr_scheduler.ReduceLROnPlateau(
        optim,mode="min",factor=cfg.scheduler_factor,patience=cfg.scheduler_patience,
        min_lr=cfg.min_lr,threshold=cfg.min_delta)
    best_loss=float("inf"); best_ep=-1; wait=0;history=[]
    ckpt=out/"checkpoints"/f"{name}.pt"
    for epoch in range(1,cfg.epochs+1):
        model.train();train_sum=0.;count=0
        for xb,yb in train_loader:
            xb=xb.to(device,non_blocking=True).unsqueeze(1);yb=yb.to(device,non_blocking=True)
            optim.zero_grad(set_to_none=True)
            logits,z=model(xb,return_embedding=True)
            loss=loss_value(logits,z,yb,lambda_proto)
            loss.backward();optim.step()
            train_sum+=loss.item()*len(yb);count+=len(yb)
        vloss,vacc=eval_celoss(model,val_loader,device)
        sched.step(vloss)
        lr=float(optim.param_groups[0]["lr"])
        record={"epoch":epoch,"train_loss":train_sum/max(count,1),"d4_query_ce":vloss,
                "d4_query_accuracy":vacc,"lr":lr}
        history.append(record)
        print(f"[{name}] epoch {epoch:02d} train {record['train_loss']:.4f} "
              f"D4_ce {vloss:.4f} D4_acc {vacc*100:.2f}% lr={lr:.2g}",flush=True)
        if vloss < best_loss-cfg.min_delta:
            best_loss=vloss;best_ep=epoch;wait=0
            write_checkpoint(ckpt,model,{"name":name,"best_epoch":best_ep,"validation_loss":vloss,
                                        "validation_accuracy":vacc,"feature": "raw" if name=="M0_raw" else "ceps"})
        else:
            wait+=1
            if wait>=cfg.early_stop_patience:
                print(f"[{name}] Early Stopping; best epoch {best_ep}",flush=True)
                break
    pd.DataFrame(history).to_csv(out/f"history_{name}.csv",index=False)
    model.load_state_dict(torch.load(ckpt,map_location="cpu",weights_only=False)["state_dict"])
    del train_loader,val_loader,x,y,vx,vy
    gc.collect()
    return model


def adaptation_clone(model, sx, sy, steps, lr, lambda_proto, device, weight_decay=0.):
    """Fine-tune clone ONLY on labeled support samples; Query never enters optimizer."""
    clone=copy.deepcopy(model).to(device)
    clone.train()
    opt=torch.optim.Adam(clone.parameters(),lr=lr,weight_decay=weight_decay)
    xt=torch.from_numpy(np.ascontiguousarray(sx,dtype=np.float32)).unsqueeze(1).to(device)
    yt=torch.from_numpy(np.asarray(sy,dtype=np.int64)).to(device)
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        logits,z=clone(xt,return_embedding=True)
        lv=loss_value(logits,z,yt,lambda_proto)
        lv.backward();opt.step()
    clone.eval()
    return clone


@torch.no_grad()
def get_predictions(model,x,device,batch_size=2048):
    model.eval(); pred=[]
    for start in range(0,len(x),batch_size):
        xt=torch.from_numpy(np.ascontiguousarray(x[start:start+batch_size],dtype=np.float32)).unsqueeze(1).to(device)
        pred.append(model(xt).argmax(1).cpu().numpy())
    return np.concatenate(pred) if pred else np.array([],dtype=int)


@torch.no_grad()
def get_ce(model,x,y,device,batch_size=2048):
    model.eval(); loss=0.;n=0
    for start in range(0,len(x),batch_size):
        xt=torch.from_numpy(np.ascontiguousarray(x[start:start+batch_size],dtype=np.float32)).unsqueeze(1).to(device)
        yt=torch.from_numpy(np.asarray(y[start:start+batch_size],dtype=np.int64)).to(device)
        loss+=nn.functional.cross_entropy(model(xt),yt,reduction="sum").item();n+=len(yt)
    return loss/max(n,1)


def meta_train(model, sessions, cfg, out, device):
    """First-order Reptile over D1-D3 per-session tasks, starting from M2 weights."""
    source=[s for s in sessions if s.day<=3]
    val=[s for s in sessions if s.day==4]
    rng=np.random.default_rng(cfg.seed+309)
    opt=torch.optim.SGD(model.parameters(),lr=cfg.meta_outer_lr)
    sched=torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,mode="min",factor=cfg.scheduler_factor,patience=2,min_lr=cfg.min_lr)
    ckpt=out/"checkpoints"/"M3_reptile.pt"
    best=float("inf");bad=0; history=[]
    model=model.to(device)
    for it in range(1,cfg.meta_iterations+1):
        task=source[int(rng.integers(len(source)))]
        sx,qx=partition_session(task,cfg.meta_support_shots,cfg.seed+it)
        # Reptile inner-loop learns from SUPPORT ONLY, and outer loop moves toward θ'.
        adapted=adaptation_clone(model,task.x[sx],task.y[sx],cfg.meta_inner_steps,
                                 cfg.meta_inner_lr,cfg.lambda_proto,device)
        old_params=[p.detach().clone() for p in model.parameters()]
        new_params=[p.detach() for p in adapted.parameters()]
        opt.zero_grad(set_to_none=True)
        for p,old,new in zip(model.parameters(),old_params,new_params):
            p.grad=old-new
        opt.step()
        del adapted,old_params,new_params
        if it==1 or it%cfg.meta_val_every==0 or it==cfg.meta_iterations:
            val_losses=[];val_accs=[]
            for s in val:
                support,query=partition_session(s,cfg.meta_support_shots,cfg.seed,
                                                query_cap=cfg.val_query_cap_per_class)
                cand=adaptation_clone(model,s.x[support],s.y[support],cfg.eval_adapt_steps,
                                      cfg.eval_adapt_lr,cfg.lambda_proto,device)
                ce=get_ce(cand,s.x[query],s.y[query],device)
                pr=get_predictions(cand,s.x[query],device)
                val_losses.append(ce)
                val_accs.append(float((pr==s.y[query]).mean()))
                del cand
            mean_loss=float(np.mean(val_losses));mean_acc=float(np.mean(val_accs))
            sched.step(mean_loss)
            entry={"iteration":it,"d4_adapt_ce":mean_loss,"d4_adapt_acc":mean_acc,
                   "meta_lr":float(opt.param_groups[0]["lr"])}
            history.append(entry)
            print(f"[M3] meta-it {it:03d} D4 adapted CE={mean_loss:.4f}, "
                  f"acc={mean_acc*100:.2f}%, outerLR={entry['meta_lr']:.4g}",flush=True)
            if mean_loss < best-cfg.min_delta:
                best=mean_loss;bad=0
                write_checkpoint(ckpt,model,{"name":"M3_reptile","best_iteration":it,
                                            "validation_ce":mean_loss,"validation_acc":mean_acc})
            else:
                bad+=1
                if bad>=cfg.meta_early_stop_patience:
                    print("[M3] Meta Early Stopping",flush=True)
                    break
    pd.DataFrame(history).to_csv(out/"history_M3_reptile.csv",index=False)
    model.load_state_dict(torch.load(ckpt,map_location="cpu",weights_only=False)["state_dict"])
    return model


def metrics(y_true,y_pred):
    return {"accuracy":float(accuracy_score(y_true,y_pred)),
            "balanced_accuracy":float(balanced_accuracy_score(y_true,y_pred)),
            "macro_f1":float(f1_score(y_true,y_pred,labels=list(range(N_CLASSES)),average="macro",zero_division=0))}


def final_eval(model,sessions,cfg,out,device,model_name,lambda_proto):
    """Evaluate *only after model frozen*: D5 same-session 0shot vs Kshot;
    extra cross-session evaluation support D5_T1 -> query all D5_T2.
    """
    rows=[];cms={}
    final=[s for s in sessions if s.day==5]
    for k in cfg.eval_shots:
        cm0=np.zeros((N_CLASSES,N_CLASSES),dtype=int)
        cm1=np.zeros_like(cm0)
        for s in final:
            support,query=partition_session(s,k,cfg.seed,query_cap=cfg.final_query_cap_per_class)
            yp0=get_predictions(model,s.x[query],device)
            adapted=adaptation_clone(model,s.x[support],s.y[support],cfg.eval_adapt_steps,
                                     cfg.eval_adapt_lr,lambda_proto,device)
            yp1=get_predictions(adapted,s.x[query],device)
            base=metrics(s.y[query],yp0); after=metrics(s.y[query],yp1)
            record={"model":model_name,"experiment":"A_within_session","subject":s.subject,
                    "day":s.day,"session":s.session,"shots_per_class":k,
                    "n_support":len(support),"n_query":len(query),
                    **{f"before_{m}":v for m,v in base.items()},
                    **{f"after_{m}":v for m,v in after.items()},
                    "delta_accuracy":after["accuracy"]-base["accuracy"]}
            rows.append(record)
            cm0+=confusion_matrix(s.y[query],yp0,labels=list(range(N_CLASSES)))
            cm1+=confusion_matrix(s.y[query],yp1,labels=list(range(N_CLASSES)))
            del adapted
        cms[f"A_k{k}_before"]=cm0;cms[f"A_k{k}_after"]=cm1
    # Test B: D5 T1 as Support, evaluate untouched D5 T2; no T2 samples used in adaptation.
    bys={(s.subject,s.session):s for s in final}
    for k in cfg.eval_shots:
        cm0=np.zeros((N_CLASSES,N_CLASSES),dtype=int);cm1=np.zeros_like(cm0)
        for subject in range(1,11):
            morning= bys[(subject,1)]; afternoon=bys[(subject,2)]
            support,_=partition_session(morning,k,cfg.seed)
            qi=np.arange(len(afternoon.x),dtype=np.int64)
            yp0=get_predictions(model,afternoon.x[qi],device)
            adapted=adaptation_clone(model,morning.x[support],morning.y[support],cfg.eval_adapt_steps,
                                     cfg.eval_adapt_lr,lambda_proto,device)
            yp1=get_predictions(adapted,afternoon.x[qi],device)
            base=metrics(afternoon.y[qi],yp0);after=metrics(afternoon.y[qi],yp1)
            rows.append({"model":model_name,"experiment":"B_morning_to_afternoon","subject":subject,
                         "day":5,"session":2,"shots_per_class":k,
                         "n_support":len(support),"n_query":len(qi),
                         **{f"before_{m}":v for m,v in base.items()},
                         **{f"after_{m}":v for m,v in after.items()},
                         "delta_accuracy":after["accuracy"]-base["accuracy"]})
            cm0+=confusion_matrix(afternoon.y[qi],yp0,labels=list(range(N_CLASSES)))
            cm1+=confusion_matrix(afternoon.y[qi],yp1,labels=list(range(N_CLASSES)))
            del adapted
        cms[f"B_k{k}_before"]=cm0;cms[f"B_k{k}_after"]=cm1
    pd.DataFrame(rows).to_csv(out/f"test_{model_name}_per_subject.csv",index=False)
    np.savez_compressed(out/f"test_{model_name}_confusion_matrices.npz",**cms)
    return rows


def run(cfg):
    start=time.time()
    seed_all(cfg.seed)
    device=torch.device("cuda" if cfg.device=="auto" and torch.cuda.is_available()
                        else "cpu" if cfg.device=="auto" else cfg.device)
    out=Path(cfg.output).expanduser().resolve();out.mkdir(parents=True,exist_ok=True)
    if cfg.channel in ("CH09","CH10"):
        raise ValueError("CH09 and CH10 are documented structurally empty DB6 channels")
    if not re.fullmatch(r"CH(0[1-8]|1[1-6])",cfg.channel):
        raise ValueError("Choose CH01..CH08, CH11..CH16")
    atomic_json(out/"run_config.json",asdict(cfg))
    print(f"Device={device}, locked channel={cfg.channel}, output={out}",flush=True)
    locations=collect_files(Path(cfg.input_root))
    manifest=check_manifest(locations,out)
    # Strict no-peek: read D1-D4 only; D5 is not opened until ALL models are frozen.
    raw_sessions=load_sessions(locations,manifest,cfg,out,allowed_days={1,2,3,4})
    # Make the two feature sets from the same single-channel DB6 sessions.
    raw=make_feature_sessions(raw_sessions,"raw",cfg,out)
    cep=make_feature_sessions(raw_sessions,"ceps",cfg,out)
    del raw_sessions
    gc.collect()

    results=[]
    models={}
    for name,sessions,lamb in [("M0_raw",raw,0.),("M1_ceps",cep,0.),("M2_proto",cep,cfg.lambda_proto)]:
        model=fit_model(name,sessions,cfg,out,device,lamb)
        models[name]=model.cpu()
        if device.type=="cuda":torch.cuda.empty_cache()

    # Reptile starts at M2's best D4-pre-adaptation checkpoint.
    meta_init=copy.deepcopy(models["M2_proto"]).to(device)
    meta_model=meta_train(meta_init,cep,cfg,out,device)
    models["M3_reptile"]=meta_model.cpu()
    if device.type=="cuda":torch.cuda.empty_cache()

    # D5 is opened only now: after all model selection, checkpoints and learning rates are frozen.
    print("All checkpoints frozen. Opening D5 for the FIRST time...",flush=True)
    d5_raw_unscaled=load_sessions(locations,manifest,cfg,out,allowed_days={5})
    d5_raw=normalize_heldout_sessions(d5_raw_unscaled,"raw",cfg,out)
    d5_cep=normalize_heldout_sessions(d5_raw_unscaled,"ceps",cfg,out)
    del d5_raw_unscaled
    gc.collect()
    # WARNING: D5 evaluation is the LAST activity; never select models using D5.
    for name in ("M0_raw","M1_ceps","M2_proto","M3_reptile"):
        s=d5_raw if name=="M0_raw" else d5_cep
        lamb=cfg.lambda_proto if name in ("M2_proto","M3_reptile") else 0.
        print(f"[FINAL D5] evaluating {name}",flush=True)
        rows=final_eval(models[name].to(device),s,cfg,out,device,name,lamb)
        results.extend(rows)
        models[name].cpu()
        if device.type=="cuda":torch.cuda.empty_cache()

    df=pd.DataFrame(results)
    df.to_csv(out/"test_all_models_per_subject.csv",index=False)
    summary=(df.groupby(["model","experiment","shots_per_class"],as_index=False)
               .agg(n_sessions=("subject","size"),
                    before_accuracy_mean=("before_accuracy","mean"),
                    after_accuracy_mean=("after_accuracy","mean"),
                    delta_accuracy_mean=("delta_accuracy","mean"),
                    delta_accuracy_sd=("delta_accuracy","std"),
                    after_macro_f1_mean=("after_macro_f1","mean"),
                    after_balanced_accuracy_mean=("after_balanced_accuracy","mean")))
    summary.to_csv(out/"FINAL_SUMMARY.csv",index=False)
    print("\n=== D5 FINAL TEST MEANS (sessions unweighted) ===",flush=True)
    print(summary.to_string(index=False),flush=True)
    atomic_json(out/"DONE.json",{"completed":True,"selected_channel":cfg.channel,
                                 "seconds":time.time()-start,"summary_csv":"FINAL_SUMMARY.csv",
                                 "note":"No model or hyperparameter selection from final D5 Query."})
    # Compact export: exclude multi-GB cached raw windows.
    bundle=out/"RESULTS_ONLY.zip"
    with zipfile.ZipFile(bundle,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
        for file in sorted(out.rglob("*")):
            if file.is_file() and file != bundle and "cache" not in file.relative_to(out).parts:
                zf.write(file,arcname=str(file.relative_to(out)))
    print(f"Saved report bundle: {bundle}",flush=True)


def parse_args():
    p=argparse.ArgumentParser(description=__doc__)
    for key,fld in Config.__dataclass_fields__.items():
        default=fld.default
        if isinstance(default,tuple):
            p.add_argument("--"+key.replace("_","-"),type=str,default=",".join(map(str,default)))
        elif isinstance(default,bool):
            p.add_argument("--"+key.replace("_","-"),action="store_false" if default else "store_true")
        else:
            p.add_argument("--"+key.replace("_","-"),type=type(default),default=default)
    args=vars(p.parse_args())
    args["eval_shots"]=tuple(int(v) for v in args["eval_shots"].split(","))
    return Config(**args)



# -------------------- USER CONFIG (EDIT HERE) --------------------
CFG = Config(
    input_root="/kaggle/input",
    output="/kaggle/working/db6_adaptation_full",
    channel="CH01",              # LOCKED: skip 14-channel selection
    seed=11,
    epochs=50,
    meta_iterations=200,
    eval_shots=(1, 2, 3),
    lr=1e-3,
    scheduler_patience=3,
    scheduler_factor=0.5,
    min_lr=1e-6,
    early_stop_patience=12,
)
run(CFG)


In [ ]:
# Copy TOAN BO vao MOT cell Kaggle.
# Dataset: trilucas/semg-db6. Accelerator: GPU.
# D1-D3: Train/Meta-train | D4: Validation | D5: Exploratory retest.
#
# CCA = mean(real(IFFT(log(power spectrum)))), INCLUDING coefficient zero.
# temporal: CCA on 16 short frames within each 200 ms window.
# paper_scalar: one CCA value for the entire 200 ms window.
#
# Temporal CCA is a single-electrode extension, not an exact reproduction
# of EMG-Adapt's multi-electrode architecture and evaluation protocol.

import copy, gc, hashlib, io, json, random, re, time, zipfile
from dataclasses import dataclass, replace
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
)

# ==================== CAU HINH ====================

C = dict(
    input_root="/kaggle/input",
    output="/kaggle/working/DB6_CH01_CCA_V2",

    channel="CH01",
    fs=2000,
    window=400,
    hop=200,

    cca_mode="temporal",       # "temporal" or "paper_scalar"
    cca_frame=100,             # 50 ms
    cca_hop=20,                # 10 ms -> 16 CCA values/window
    eps=1e-12,

    train_seeds=(11,),         # Stability study: (11,22,33,44,55)
    support_seeds=(101,202,303),
    shots=(1,2,3),

    batch_size=256,
    epochs=40,
    lr=1e-3,
    weight_decay=1e-4,

    lambda_proto=0.25,
    lr_patience=3,
    early_patience=12,
    min_delta=1e-4,

    meta_iterations=200,
    meta_shots=3,
    inner_steps=25,
    inner_lr=1e-3,
    outer_lr=0.5,
    meta_val_every=25,
    meta_patience=5,

    adapt_steps=25,
    adapt_lr=1e-3,

    raw_baseline=True,
    cca_ablation=True,
    evaluate_d5=True,
    resume=True,

    # Reuse compatible raw-window cache from the previous notebook.
    legacy_cache="/kaggle/working/db6_adaptation_full/cache",
    device="auto",
)

VERSION = "CH01_CCA_V2_20261029a"

LABELS = {
    0:0, 1:1, 3:2, 4:3,
    6:4, 9:5, 10:6, 11:7,
}

NAMES = [
    "REST",
    "Large Diameter",
    "Adducted Thumb",
    "Index Finger Extension",
    "Medium Wrap",
    "Writing Tripod",
    "Power Sphere",
    "Precision Sphere",
]

NC = len(NAMES)
RX = re.compile(r"S(\d+)_D([1-5])_T([12])\.mat$", re.I)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def digest(obj):
    text = json.dumps(obj, sort_keys=True, default=str)
    return hashlib.sha256(text.encode()).hexdigest()[:20]


def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(obj, ensure_ascii=False, indent=2, default=str),
        encoding="utf8",
    )
    tmp.replace(path)


@dataclass
class Session:
    subject: int
    day: int
    trial: int
    x: np.ndarray
    y: np.ndarray
    groups: np.ndarray

    @property
    def key(self):
        return (self.subject, self.day, self.trial)


def inventory(root, out):
    found = {}

    for p in sorted(Path(root).rglob("*.mat")):
        if not RX.fullmatch(p.name):
            continue

        if p.name in found and found[p.name]["path"] != str(p):
            raise RuntimeError(
                f"Duplicate dataset filename: {p.name}; restrict input_root."
            )

        found[p.name] = dict(
            path=str(p),
            member=None,
            size=p.stat().st_size,
        )

    for p in sorted(Path(root).rglob("*.zip")):
        with zipfile.ZipFile(p) as z:
            for info in z.infolist():
                name = Path(info.filename).name

                if RX.fullmatch(name) and name not in found:
                    found[name] = dict(
                        path=str(p),
                        member=info.filename,
                        size=info.file_size,
                    )

    expected = {
        f"S{s}_D{d}_T{t}.mat"
        for s in range(1, 11)
        for d in range(1, 6)
        for t in (1, 2)
    }

    missing = expected - set(found) - {"S2_D2_T2.mat"}

    if missing:
        raise RuntimeError(f"Missing DB6 files: {sorted(missing)}")

    rows = []

    for name in sorted(expected & set(found)):
        s, d, t = map(int, RX.fullmatch(name).groups())

        rows.append(dict(
            file=name,
            subject=s,
            day=d,
            trial=t,
            **found[name],
            usable=name != "S2_D2_T2.mat",
        ))

    pd.DataFrame(rows).to_csv(out / "manifest.csv", index=False)

    print(
        f"Manifest: {len(rows)} files; "
        "exclude known incomplete S2_D2_T2.mat.",
        flush=True,
    )

    return rows


def window_signal(signal, labels, repetitions, window, hop):
    # Windows never cross label/repetition boundaries.
    # All windows from a repetition share one group.

    cuts = np.r_[
        0,
        np.flatnonzero(
            (labels[1:] != labels[:-1]) |
            (repetitions[1:] != repetitions[:-1])
        ) + 1,
        len(labels),
    ]

    xs, ys, gs = [], [], []

    for segment, (a, b) in enumerate(zip(cuts[:-1], cuts[1:])):
        lab = int(labels[a])

        if lab not in LABELS or b - a < window:
            continue

        cls = LABELS[lab]
        starts = np.arange(a, b - window + 1, hop)

        xs.append(
            signal[starts[:, None] + np.arange(window)].astype(np.float32)
        )
        ys.append(np.full(len(starts), cls, np.int64))

        if cls and repetitions[a] > 0:
            group = cls * 100000 + int(repetitions[a])
        else:
            group = 1000000 + segment

        gs.append(np.full(len(starts), group, np.int64))

    if not xs:
        raise ValueError("No valid windows.")

    return (
        np.concatenate(xs),
        np.concatenate(ys),
        np.concatenate(gs),
    )


def load_sessions(rows, days, c, out):
    sessions = []
    cache = out / "cache"
    cache.mkdir(exist_ok=True)

    for row in rows:
        if not row["usable"] or row["day"] not in days:
            continue

        name = row["file"]
        stem = Path(name).stem

        key = digest([
            VERSION,
            c["channel"],
            c["window"],
            c["hop"],
            row["size"],
        ])

        target = cache / f"{stem}_{key}.npz"

        legacy = Path(c["legacy_cache"]) / (
            f"{stem}_{c['channel']}"
            f"_w{c['window']}_h{c['hop']}_b{row['size']}.npz"
        )

        cached = target if target.exists() else legacy

        if cached.exists():
            with np.load(cached, allow_pickle=False) as z:
                x = z["x"].astype(np.float32)
                y = z["y"].astype(np.int64)
                g = z["g"].astype(np.int64)

        else:
            keys = ["emg", "restimulus", "rerepetition"]

            if row["member"] is None:
                mat = loadmat(row["path"], variable_names=keys)
            else:
                with zipfile.ZipFile(row["path"]) as z:
                    mat = loadmat(
                        io.BytesIO(z.read(row["member"])),
                        variable_names=keys,
                    )

            emg = np.asarray(mat["emg"])
            labels = mat["restimulus"].ravel()
            reps = mat["rerepetition"].ravel()

            if (
                emg.ndim != 2 or
                emg.shape[1] != 16 or
                len(labels) != len(emg) or
                len(reps) != len(emg)
            ):
                raise ValueError(f"Invalid dimensions: {name}")

            channel_index = int(c["channel"][2:]) - 1

            x, y, g = window_signal(
                emg[:, channel_index],
                labels,
                reps,
                c["window"],
                c["hop"],
            )

            np.savez_compressed(target, x=x, y=y, g=g)
            del mat, emg, labels, reps

        if (
            x.shape != (len(y), c["window"]) or
            len(g) != len(y) or
            not np.isfinite(x).all()
        ):
            raise ValueError(f"Invalid windows/cache: {name}")

        if set(np.unique(y)) != set(range(NC)):
            raise ValueError(f"Missing classes: {name}")

        sessions.append(Session(
            row["subject"],
            row["day"],
            row["trial"],
            x, y, g,
        ))

        if len(sessions) % 10 == 0:
            print(
                f"Loaded {len(sessions)} sessions "
                f"from days {sorted(days)}",
                flush=True,
            )

    return sessions


def cca_features(x, c):
    # Both modes perform the AVERAGE step.
    # No truncation of cepstral coefficients.
    #
    # NumPy DFT identity:
    # mean(real(ifft(log(P)))) = log(P[0]) / N.
    #
    # Therefore full-cepstrum averaging is a scalar descriptor.
    # Do not demean each frame before applying this formula.

    result = []

    for start in range(0, len(x), 512):
        a = x[start:start + 512].astype(np.float64)

        if c["cca_mode"] == "temporal":
            a = np.lib.stride_tricks.sliding_window_view(
                a, c["cca_frame"], axis=-1
            )[:, ::c["cca_hop"], :]
        else:
            a = a[:, None, :]

        power = np.abs(np.fft.rfft(a, axis=-1)) ** 2

        q = np.fft.irfft(
            np.log(np.maximum(power, c["eps"])),
            n=a.shape[-1],
            axis=-1,
        )

        result.append(
            q.mean(axis=-1).astype(np.float32)
        )

    return np.ascontiguousarray(np.concatenate(result))


def build_plans(sessions, c):
    # Reserve MAX_K groups for support at EVERY K.
    # Query is identical across K, models and training seeds.
    # Support sets are nested.

    plans = {}
    kmax = max(max(c["shots"]), c["meta_shots"])

    for s in sessions:
        for draw in c["support_seeds"]:
            rng = np.random.default_rng(
                draw +
                s.subject * 1009 +
                s.day * 101 +
                s.trial * 17
            )

            support = np.empty((NC, kmax), np.int64)
            query = []

            for cls in range(NC):
                ix = np.flatnonzero(s.y == cls)
                groups = np.unique(s.groups[ix])

                if len(groups) <= kmax:
                    raise ValueError(
                        f"Too few groups: {s.key}, class {cls}"
                    )

                reserve = rng.permutation(groups)[:kmax]

                for j, gid in enumerate(reserve):
                    candidates = ix[s.groups[ix] == gid]
                    support[cls, j] = rng.choice(candidates)

                query.extend(
                    ix[~np.isin(s.groups[ix], reserve)].tolist()
                )

            query = np.asarray(query, np.int64)

            overlap = np.intersect1d(
                s.groups[support.ravel()],
                s.groups[query],
            )

            if overlap.size:
                raise AssertionError("Support/query group overlap")

            plans[s.key, draw] = (support, query)

    return plans


def fit_scaler(sessions, scalar=False):
    src = [s.x for s in sessions if s.day <= 3]

    total = sum(
        x.size if scalar else len(x)
        for x in src
    )

    axis = None if scalar else 0

    mu = sum(
        np.asarray(x, dtype=np.float64).sum(axis=axis)
        for x in src
    ) / total

    var = sum(
        np.square(
            np.asarray(x, dtype=np.float64) - mu
        ).sum(axis=axis)
        for x in src
    ) / total

    return (
        np.asarray(mu, np.float32),
        np.asarray(
            np.maximum(np.sqrt(var), 1e-8),
            np.float32,
        ),
    )


def features(raw, c, out, scalers=None):
    sets = {
        "cca": [
            replace(s, x=cca_features(s.x, c))
            for s in raw
        ]
    }

    if c["raw_baseline"]:
        sets["raw"] = raw

    if scalers is None:
        scalers = {
            kind: fit_scaler(
                sessions,
                scalar=kind == "raw",
            )
            for kind, sessions in sets.items()
        }

        np.savez(
            out / "normalizers.npz",
            **{
                f"{kind}_{name}": value
                for kind, pair in scalers.items()
                for name, value in zip(("mean", "std"), pair)
            },
        )

        if float(np.max(scalers["cca"][1])) <= 1e-8:
            raise ValueError(
                "CCA has collapsed to a constant; "
                "inspect signal/DC and numerical floor."
            )

    for kind, sessions in sets.items():
        mu, sd = scalers[kind]

        sets[kind] = [
            replace(
                s,
                x=np.asarray((s.x - mu) / sd, np.float32),
            )
            for s in sessions
        ]

    return sets, scalers


class Net(nn.Module):
    def __init__(self, length):
        super().__init__()

        blocks = []
        a = 1
        remaining = length

        for b, k in zip(
            (32, 64, 128, 128),
            (7, 5, 3, 3),
        ):
            k = k if remaining > 1 else 1

            blocks += [
                nn.Conv1d(
                    a, b, k,
                    padding=k // 2,
                    bias=False,
                ),
                nn.BatchNorm1d(b),
                nn.ReLU(),
            ]

            if remaining >= 2:
                blocks.append(nn.MaxPool1d(2))
                remaining //= 2

            a = b

        self.conv = nn.Sequential(
            *blocks,
            nn.AdaptiveAvgPool1d(1),
        )

        self.embedding = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 128),
        )

        self.head = nn.Linear(128, NC)

        def init(m):
            if isinstance(m, (nn.Linear, nn.Conv1d)):
                nn.init.xavier_uniform_(m.weight)

                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        self.apply(init)

    def forward(self, x):
        z = self.embedding(self.conv(x))
        return self.head(z), z


def objective(logits, z, y, lam):
    # CrossEntropyLoss applies log-softmax internally.
    ce = nn.functional.cross_entropy(logits, y)

    if not lam:
        return ce

    # Euclidean distance to the own-class centroid.
    # Centroids are recomputed at every batch/support update.
    proto = z.sum() * 0

    for cls in y.unique():
        pts = z[y == cls]

        proto = proto + torch.linalg.vector_norm(
            pts - pts.mean(0, keepdim=True),
            dim=1,
        ).sum()

    return (
        (1 - lam) * ce +
        lam * proto / len(y)
    )


def adapt(model, x, y, steps, lr, lam, device):
    clone = copy.deepcopy(model).to(device)

    # Freeze BN running statistics and disable Dropout.
    # eval() does NOT disable gradients.
    # Same policy in Reptile and final adaptation.
    clone.eval()

    opt = torch.optim.Adam(clone.parameters(), lr=lr)

    xt = torch.from_numpy(
        np.ascontiguousarray(x)
    ).unsqueeze(1).to(device)

    yt = torch.from_numpy(
        np.asarray(y, np.int64)
    ).to(device)

    for _ in range(steps):
        opt.zero_grad(set_to_none=True)

        logits, z = clone(xt)
        loss = objective(logits, z, yt, lam)

        if not torch.isfinite(loss):
            raise FloatingPointError(
                "Non-finite adaptation loss"
            )

        loss.backward()
        opt.step()

    return clone


@torch.no_grad()
def predict(model, x, y, device):
    model.eval()
    predicted = []
    total = 0.

    for a in range(0, len(x), 1024):
        xt = torch.from_numpy(
            np.ascontiguousarray(x[a:a + 1024])
        ).unsqueeze(1).to(device)

        yt = torch.from_numpy(
            np.asarray(y[a:a + 1024], np.int64)
        ).to(device)

        logits, _ = model(xt)

        total += nn.functional.cross_entropy(
            logits,
            yt,
            reduction="sum",
        ).item()

        # argmax(logits) == argmax(softmax(logits))
        predicted.append(
            logits.argmax(1).cpu().numpy()
        )

    return (
        np.concatenate(predicted),
        total / len(y),
    )


def score(y, p):
    return dict(
        acc=float(accuracy_score(y, p)),
        ba=float(balanced_accuracy_score(y, p)),
        f1=float(f1_score(
            y, p,
            labels=np.arange(NC),
            average="macro",
            zero_division=0,
        )),
    )


def validation(
    model, sessions, plans, c, device, adapted=False
):
    losses, accs = [], []

    for s in sessions:
        support, q = plans[
            s.key, c["support_seeds"][0]
        ]

        candidate = model

        if adapted:
            ix = support[:, :c["meta_shots"]].ravel()

            candidate = adapt(
                model,
                s.x[ix],
                s.y[ix],
                c["adapt_steps"],
                c["adapt_lr"],
                c["lambda_proto"],
                device,
            )

        p, loss = predict(
            candidate, s.x[q], s.y[q], device
        )

        losses.append(loss)
        accs.append(float((p == s.y[q]).mean()))

        if adapted:
            del candidate

    return (
        float(np.mean(losses)),
        float(np.mean(accs)),
    )


def checkpoint(path, model, signature, info):
    temp = path.with_suffix(".pt.tmp")

    torch.save(
        dict(
            signature=signature,
            state={
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            },
            info=info,
        ),
        temp,
    )

    temp.replace(path)


def restore(path, length, signature, device):
    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    if payload["signature"] != signature:
        raise ValueError(
            f"Checkpoint/config mismatch: {path}"
        )

    model = Net(length).to(device)
    model.load_state_dict(payload["state"])
    model.eval()

    return model


def fit(
    name, sessions, plans, c,
    seed, signature, out, device, lam
):
    path = (
        out / "checkpoints" /
        f"{name}_seed{seed}.pt"
    )

    length = sessions[0].x.shape[1]
    done = path.with_suffix(".done.json")

    if c["resume"] and path.exists() and done.exists():
        if json.loads(
            done.read_text()
        )["signature"] != signature:
            raise ValueError(
                "Use a new output folder after changing config."
            )

        print("RESUME", path.name, flush=True)

        return (
            restore(path, length, signature, device),
            path,
        )

    done.unlink(missing_ok=True)

    seed_all(seed)
    model = Net(length).to(device)

    source = [s for s in sessions if s.day <= 3]
    val = [s for s in sessions if s.day == 4]

    x = np.concatenate([s.x for s in source])
    y = np.concatenate([s.y for s in source])

    batch = min(c["batch_size"], len(y))

    loader = DataLoader(
        TensorDataset(
            torch.from_numpy(x),
            torch.from_numpy(y),
        ),
        batch_size=batch,
        shuffle=True,
        drop_last=len(y) > batch,
        num_workers=0,
        pin_memory=device.type == "cuda",
    )

    opt = torch.optim.Adam(
        model.parameters(),
        lr=c["lr"],
        weight_decay=c["weight_decay"],
    )

    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="min",
        factor=0.5,
        patience=c["lr_patience"],
        threshold=c["min_delta"],
        threshold_mode="abs",
        min_lr=1e-6,
    )

    best = float("inf")
    bad = 0
    history = []

    print(
        f"[{name}, seed={seed}] "
        f"source={x.shape}, "
        f"parameters={sum(p.numel() for p in model.parameters()):,}",
        flush=True,
    )

    for epoch in range(1, c["epochs"] + 1):
        model.train()
        total = 0.
        n = 0

        for xb, yb in loader:
            xb = xb.unsqueeze(1).to(device)
            yb = yb.to(device)

            opt.zero_grad(set_to_none=True)

            logits, z = model(xb)
            loss = objective(logits, z, yb, lam)

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    "Non-finite training loss"
                )

            loss.backward()
            opt.step()

            total += loss.item() * len(yb)
            n += len(yb)

        vl, va = validation(
            model, val, plans, c, device
        )

        sched.step(vl)

        row = dict(
            epoch=epoch,
            train_loss=total / n,
            d4_ce=vl,
            d4_acc=va,
            lr=opt.param_groups[0]["lr"],
        )

        history.append(row)

        print(
            f"  epoch {epoch:02d}: "
            f"train={total/n:.4f}, "
            f"D4 CE={vl:.4f}, "
            f"acc={va:.2%}, "
            f"LR={row['lr']:.2g}",
            flush=True,
        )

        if vl < best - c["min_delta"]:
            best = vl
            bad = 0
            checkpoint(path, model, signature, row)
        else:
            bad += 1

            if bad >= c["early_patience"]:
                break

    pd.DataFrame(history).to_csv(
        out / f"history_{name}_seed{seed}.csv",
        index=False,
    )

    save_json(
        done,
        dict(signature=signature, completed=True),
    )

    del loader, x, y, model
    gc.collect()

    return (
        restore(path, length, signature, device),
        path,
    )


def reptile(
    model, sessions, plans, c,
    seed, signature, out, device
):
    path = (
        out / "checkpoints" /
        f"M3_CCA_Reptile_seed{seed}.pt"
    )

    length = sessions[0].x.shape[1]
    done = path.with_suffix(".done.json")

    if c["resume"] and path.exists() and done.exists():
        if json.loads(
            done.read_text()
        )["signature"] != signature:
            raise ValueError(
                "Use a new output folder after changing config."
            )

        return (
            restore(path, length, signature, device),
            path,
        )

    done.unlink(missing_ok=True)

    seed_all(seed + 9000)
    rng = np.random.default_rng(seed + 9000)

    source = [s for s in sessions if s.day <= 3]
    val = [s for s in sessions if s.day == 4]

    best, acc = validation(
        model, val, plans, c, device, adapted=True
    )

    # Iteration zero is a valid candidate.
    # Do not force a worse meta-checkpoint.
    history = [
        dict(
            iteration=0,
            d4_adapt_ce=best,
            d4_adapt_acc=acc,
            beta=0.,
        )
    ]

    checkpoint(
        path, model, signature, history[-1]
    )

    bad = 0

    for it in range(1, c["meta_iterations"] + 1):
        task = source[
            int(rng.integers(len(source)))
        ]

        ids = []

        for cls in range(NC):
            ix = np.flatnonzero(task.y == cls)
            groups = np.unique(task.groups[ix])

            chosen = rng.choice(
                groups,
                c["meta_shots"],
                replace=False,
            )

            for gid in chosen:
                candidates = ix[
                    task.groups[ix] == gid
                ]
                ids.append(rng.choice(candidates))

        ids = np.asarray(ids, np.int64)

        local = adapt(
            model,
            task.x[ids],
            task.y[ids],
            c["inner_steps"],
            c["inner_lr"],
            c["lambda_proto"],
            device,
        )

        beta = c["outer_lr"] * (
            1 - it / c["meta_iterations"]
        )

        with torch.no_grad():
            for p, q in zip(
                model.parameters(),
                local.parameters(),
            ):
                p.add_(q - p, alpha=beta)

        del local

        if (
            it == 1 or
            it % c["meta_val_every"] == 0 or
            it == c["meta_iterations"]
        ):
            loss, acc = validation(
                model, val, plans, c,
                device, adapted=True,
            )

            row = dict(
                iteration=it,
                d4_adapt_ce=loss,
                d4_adapt_acc=acc,
                beta=beta,
            )

            history.append(row)

            print(
                f"[Reptile, seed={seed}] "
                f"iteration={it}, "
                f"D4 adapted CE={loss:.4f}, "
                f"acc={acc:.2%}, "
                f"beta={beta:.4f}",
                flush=True,
            )

            if loss < best - c["min_delta"]:
                best = loss
                bad = 0

                checkpoint(
                    path, model, signature, row
                )
            else:
                bad += 1

                if bad >= c["meta_patience"]:
                    break

    pd.DataFrame(history).to_csv(
        out / f"history_Reptile_seed{seed}.csv",
        index=False,
    )

    save_json(
        done,
        dict(signature=signature, completed=True),
    )

    return (
        restore(path, length, signature, device),
        path,
    )


def evaluate(
    model, sessions, plans, c,
    seed, name, split, device, lam
):
    rows = []
    cms = {}
    bykey = {s.key: s for s in sessions}

    def episode(
        s, qsession, pool, q, draw, experiment
    ):
        p0, _ = predict(
            model,
            qsession.x[q],
            qsession.y[q],
            device,
        )

        before = score(qsession.y[q], p0)

        for k in c["shots"]:
            ix = pool[:, :k].ravel()

            local = adapt(
                model,
                s.x[ix],
                s.y[ix],
                c["adapt_steps"],
                c["adapt_lr"],
                lam,
                device,
            )

            p1, _ = predict(
                local,
                qsession.x[q],
                qsession.y[q],
                device,
            )

            after = score(qsession.y[q], p1)

            rows.append(dict(
                split=split,
                model=name,
                train_seed=seed,
                support_seed=draw,
                experiment=experiment,
                subject=s.subject,
                trial=qsession.trial,
                shots=k,
                n_support=len(ix),
                n_query=len(q),

                **{
                    f"before_{metric}": value
                    for metric, value in before.items()
                },

                **{
                    f"after_{metric}": value
                    for metric, value in after.items()
                },

                delta_acc=after["acc"] - before["acc"],
            ))

            for phase, pred in (
                ("before", p0),
                ("after", p1),
            ):
                key = f"{experiment}_K{k}_{phase}"

                cm = confusion_matrix(
                    qsession.y[q],
                    pred,
                    labels=np.arange(NC),
                )

                cms[key] = cms.get(
                    key,
                    np.zeros((NC, NC), np.int64),
                ) + cm

            del local

    for s in sessions:
        for draw in c["support_seeds"]:
            pool, q = plans[s.key, draw]

            episode(
                s, s, pool, q, draw,
                "A_within_session",
            )

            if s.trial == 1:
                afternoon = bykey[
                    s.subject, s.day, 2
                ]

                episode(
                    s,
                    afternoon,
                    pool,
                    np.arange(len(afternoon.y)),
                    draw,
                    "B_T1_to_T2",
                )

    return rows, cms


def self_check(c):
    rng = np.random.default_rng(7)
    a = rng.normal(size=(4, c["window"]))

    cc = dict(c, cca_mode="paper_scalar")

    expected = np.log(
        np.maximum(
            np.square(a.sum(-1)),
            c["eps"],
        )
    ) / a.shape[-1]

    np.testing.assert_allclose(
        cca_features(a, cc).ravel(),
        expected,
        rtol=2e-6,
        atol=1e-8,
    )

    for length in (1, 16, 400):
        model = Net(length).eval()

        logits, z = model(
            torch.zeros(2, 1, length)
        )

        assert logits.shape == (2, NC)
        assert z.shape == (2, 128)

    print(
        "Self-check OK: CCA formula, "
        "scalar/temporal/raw network dimensions.",
        flush=True,
    )


def main(config):
    c = copy.deepcopy(config)
    out = Path(c["output"])
    out.mkdir(parents=True, exist_ok=True)

    (out / "checkpoints").mkdir(exist_ok=True)

    if c["channel"] != "CH01":
        raise ValueError(
            "This experiment locks CH01."
        )

    if c["cca_mode"] not in (
        "temporal", "paper_scalar"
    ):
        raise ValueError("Invalid cca_mode")

    if (
        c["window"] < c["cca_frame"] or
        min(
            c["cca_frame"],
            c["cca_hop"],
            c["hop"],
        ) < 1
    ):
        raise ValueError("Invalid window sizes")

    if (
        not c["train_seeds"] or
        not c["support_seeds"] or
        min(c["shots"]) < 1
    ):
        raise ValueError("Empty seeds/invalid shots")

    if min(
        c["epochs"],
        c["meta_iterations"],
        c["inner_steps"],
        c["adapt_steps"],
    ) < 1:
        raise ValueError(
            "Use positive update counts"
        )

    device = torch.device(
        (
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )
        if c["device"] == "auto"
        else c["device"]
    )

    print(
        f"Device={device}; CH01; "
        f"CCA mode={c['cca_mode']}; "
        f"8 classes; "
        f"window={c['window']/c['fs']:.3f}s",
        flush=True,
    )

    self_check(c)
    rows = inventory(c["input_root"], out)

    signature = digest(dict(
        version=VERSION,

        config={
            key: value
            for key, value in c.items()
            if key not in ("resume", "device")
        },

        manifest=rows,
    ))

    settings = dict(
        version=VERSION,
        signature=signature,
        config=c,
        classes=NAMES,
        torch_version=torch.__version__,
        normalizer_fit_days=[1,2,3],

        cca_note=(
            "Temporal CCA is a single-electrode extension. "
            "paper_scalar averages the full real cepstrum."
        ),

        preprocessing=(
            "Input as supplied; no additional 1-Hz "
            "low-pass filter; no per-frame demeaning."
        ),

        one_shot_note=(
            "Own-class centroid loss is zero with "
            "one support sample/class; support updates "
            "then use CE only."
        ),

        d5_status=(
            "exploratory_retest: D5 results were "
            "inspected in the previous experiment"
        ),

        summary_note=(
            "Session/draw means within each train seed, "
            "then mean/SD across training seeds; "
            "SD undefined for one train seed."
        ),
    )

    save_json(
        out / "run_config.json",
        settings,
    )

    # Only D1-D4 signals are opened here.
    raw = load_sessions(
        rows, {1,2,3,4}, c, out
    )

    plans = build_plans(raw, c)

    feature_sets, scalers = features(
        raw, c, out
    )

    del raw
    gc.collect()

    specs = []

    if c["raw_baseline"]:
        specs.append(
            ("M0_Raw_CE", "raw", 0.)
        )

    if c["cca_ablation"]:
        specs.append(
            ("M1_CCA_CE", "cca", 0.)
        )

    specs.append((
        "M2_CCA_Prototype",
        "cca",
        c["lambda_proto"],
    ))

    records = []

    for seed in c["train_seeds"]:
        for name, kind, lam in specs:
            model, path = fit(
                name,
                feature_sets[kind],
                plans,
                c,
                seed,
                signature,
                out,
                device,
                lam,
            )

            records.append((
                seed, name, kind, lam, path
            ))

            if name == "M2_CCA_Prototype":
                meta, mpath = reptile(
                    copy.deepcopy(model),
                    feature_sets[kind],
                    plans,
                    c,
                    seed,
                    signature,
                    out,
                    device,
                )

                records.append((
                    seed,
                    "M3_CCA_Reptile",
                    kind,
                    lam,
                    mpath,
                ))

                del meta

            del model

            if device.type == "cuda":
                torch.cuda.empty_cache()

    save_json(
        out / "CHECKPOINTS_LOCKED.json",
        dict(
            signature=signature,
            checkpoints=[
                str(record[-1])
                for record in records
            ],
        ),
    )

    # ALL models and seeds have now been frozen.
    partitions = [(
        "D4",

        {
            kind: [
                s for s in sessions
                if s.day == 4
            ]
            for kind, sessions in feature_sets.items()
        },

        plans,
    )]

    del feature_sets
    gc.collect()

    if c["evaluate_d5"]:
        print(
            "Opening D5 after checkpoint selection; "
            "exploratory retest.",
            flush=True,
        )

        raw5 = load_sessions(
            rows, {5}, c, out
        )

        plans5 = build_plans(raw5, c)

        test_sets, _ = features(
            raw5,
            c,
            out,
            scalers=scalers,
        )

        del raw5
        gc.collect()

        partitions.append((
            "D5",
            test_sets,
            plans5,
        ))

    all_rows = []

    for split, sets, partition_plans in partitions:
        for seed, name, kind, lam, path in records:
            print(
                f"Evaluate {split}, "
                f"{name}, seed={seed}",
                flush=True,
            )

            model = restore(
                path,
                sets[kind][0].x.shape[1],
                signature,
                device,
            )

            result, cms = evaluate(
                model,
                sets[kind],
                partition_plans,
                c,
                seed,
                name,
                split,
                device,
                lam,
            )

            all_rows.extend(result)

            pd.DataFrame(all_rows).to_csv(
                out / "per_session_draw.csv",
                index=False,
            )

            np.savez_compressed(
                out /
                f"confusion_{split}_{name}_seed{seed}.npz",
                **cms,
            )

            del model

            if device.type == "cuda":
                torch.cuda.empty_cache()

    df = pd.DataFrame(all_rows)

    metrics = [
        "before_acc",
        "after_acc",
        "delta_acc",
        "before_ba",
        "after_ba",
        "before_f1",
        "after_f1",
    ]

    byseed = df.groupby(
        [
            "split",
            "model",
            "experiment",
            "shots",
            "train_seed",
        ],
        as_index=False,
    )[metrics].mean()

    byseed.to_csv(
        out / "summary_by_train_seed.csv",
        index=False,
    )

    group = byseed.groupby([
        "split",
        "model",
        "experiment",
        "shots",
    ])

    summary = group[metrics].agg([
        "mean", "std"
    ])

    summary.columns = [
        f"{a}_{b}"
        for a, b in summary.columns
    ]

    summary = summary.reset_index().merge(
        group.size()
        .rename("n_train_seeds")
        .reset_index()
    )

    summary.to_csv(
        out / "FINAL_SUMMARY.csv",
        index=False,
    )

    display = summary[[
        "split",
        "model",
        "experiment",
        "shots",
        "n_train_seeds",
        "before_acc_mean",
        "after_acc_mean",
        "delta_acc_mean",
        "after_ba_mean",
    ]].copy()

    for name in (
        "before_acc_mean",
        "after_acc_mean",
        "delta_acc_mean",
        "after_ba_mean",
    ):
        display[name] *= 100

    print(
        "\nAccuracy/BA in %, delta in percentage points; "
        "query fixed across K:\n",
        display.round(3).to_string(index=False),
        flush=True,
    )

    save_json(
        out / "DONE.json",
        dict(
            completed=True,
            signature=signature,
            rows=len(df),
        ),
    )

    bundle = out / "RESULTS_ONLY.zip"

    with zipfile.ZipFile(
        bundle,
        "w",
        zipfile.ZIP_DEFLATED,
    ) as z:
        for p in sorted(out.rglob("*")):
            if (
                p.is_file() and
                p != bundle and
                "cache" not in p.relative_to(out).parts
            ):
                z.write(
                    p,
                    p.relative_to(out),
                )

    print(
        f"\nDONE: {bundle}",
        flush=True,
    )

    return summary


SUMMARY = main(C)